<div align="center">

# WEEK 02
## DATA CLEANING, STANDARDISATION & DATA QUALITY VALIDATION

### CadetX Virtual Work Experience
### Heavy Supplier, Inventory & Warehouse Analytics

</div>

---

| | |
|---|---|
| **Prepared by** | Keren Oyem |
| **Role** | Data Analyst |
| **Sprint** | Week 2 · Fri 30 Oct – Thu 5 Nov 2026 |
| **Catalogue tasks** | A2 Data Cleaning & Quality Improvement · A5 Data Validation & Consistency Checks · G4 Data Integrity & Error Detection |
| **Data source** | `warehouse.duckdb` (12 tables, opened **read-only**) |
| **Tools** | Python, pandas, SQL on DuckDB, pytest, Parquet |

## Project description

Week 1 profiled the twelve warehouse tables and produced a first list of data-quality concerns. Week 2 turns those concerns into evidence and decisions. The goal is a **validated, reproducible clean data layer** that Weeks 3–12 (star schema, KPIs, forecasting, dashboards) can trust.

This single notebook documents all six Week 2 tasks, in order:

| Task | What it does | Output |
|---|---|---|
| **1** | Type casting and standardisation | Numeric capacity, split dimensions, volume (m³), text postal codes, date and category review |
| **2** | Stock position reconstruction and reconciliation | Ledger rebuilt from IN / OUT / ADJUSTMENT and compared with `running_balance` and `current_stock` |
| **3** | Financial reconciliation | Line arithmetic, GST, header vs lines, invoices vs payments, customer purchase value |
| **4** | Referential integrity and business-rule checks | Primary keys, 17 foreign keys, cancelled POs, delivery dates, customer regions |
| **5** | Automated checks and data export | Pytest suite, reusable `src/cleaning.py`, cleaned Parquet files in `data/processed/` |
| **6** | Issue log and resolution decisions | Every issue given a decision: **Fix · Flag · Accept · Exclude**; sprint review and retrospective |

## How each task is laid out

1. **Objective and approach**: what the task is for and how it is done.
2. **Code cells**: each followed by a **"Reading the output"** note explaining what the result means.
3. **Task summary**: findings and the decision taken.

> **About the numbers in the notes.** Figures quoted in "Reading the output" notes come from the results recorded when Tasks 1–4 were run and from the Week 1 profile. After you re-run the notebook, the numbers on screen are the authority. If they differ from a note, trust the screen and update the note.

## Data preservation

The raw database is never modified. All cleaning happens in memory (pandas) and is written to **separate** Parquet files. No row is deleted anywhere: problems are marked with flag columns so every decision stays visible and reversible.

## Contents

0. Setup · 1. Task 1 · 2. Task 2 · 3. Task 3 · 4. Task 4 · 5. Task 5 · 6. Task 6 · 7. Week 2 wrap-up · Appendix

## 0. Setup

**Objective.** Load the libraries, define the shared settings and open the database.

**Approach.** Three small cells:
* **Settings**: database path, the money tolerance (differences under 0.01 rupee count as equal, so floating-point noise is not reported as an error) and the spelling used for cancelled statuses.
* **Rule lists**: the date and category columns reviewed in Task 1, and the candidate foreign keys and primary keys used in Task 4.
* **Connection**: opened with `read_only=True`, so no cell in this notebook can change the raw data.

Run Jupyter from the folder that contains `warehouse.duckdb`.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DB_PATH = "warehouse.duckdb"       # run Jupyter from the folder that contains this file
MONEY_TOLERANCE = 0.01             # rupees; smaller differences count as equal
CANCELLED = "('cancelled', 'canceled')"

**Reading the output.** No output is expected. The cell imports the libraries and defines the settings used by every later cell.

In [2]:
# Task 1: date columns reviewed
DATE_COLUMNS = {
    "products": ["last_purchase_date"],
    "customers": ["customer_since", "last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "purchase_orders_header": ["order_date", "expected_delivery_date", "received_date"],
    "sales_orders_header": ["order_date", "delivery_date"],
    "stock_ledger": ["movement_date"],
}


# Task 1: categorical columns reviewed
CATEGORICAL_COLUMNS = {
    "branches": ["warehouse_type"],
    "products": ["category", "machine_type"],
    "suppliers": ["supplier_type"],
    "customers": ["customer_type"],
    "invoices": ["payment_status"],
    "purchase_orders_header": ["po_status"],
    "sales_orders_header": ["order_status", "sales_channel"],
    "payments": ["payment_method"],
    "stock_ledger": ["movement_type"],
}

**Reading the output.** No output is expected. `DATE_COLUMNS` lists the 12 date columns in 7 tables; `CATEGORICAL_COLUMNS` lists the 11 label columns in 9 tables.

In [3]:
# Task 4 part 1: candidate foreign keys (parent_table, parent_key, child_table, child_key)
FK_RELATIONSHIPS = [
    ("branches", "branch_id", "inventory_master", "branch_id"),
    ("products", "product_id", "inventory_master", "product_id"),
    ("suppliers", "supplier_id", "purchase_orders_header", "supplier_id"),
    ("branches", "branch_id", "purchase_orders_header", "branch_id"),
    ("purchase_orders_header", "po_id", "purchase_orders_lines", "po_id"),
    ("products", "product_id", "purchase_orders_lines", "product_id"),
    ("customers", "customer_id", "sales_orders_header", "customer_id"),
    ("branches", "branch_id", "sales_orders_header", "branch_id"),
    ("sales_orders_header", "so_id", "sales_orders_lines", "so_id"),
    ("products", "product_id", "sales_orders_lines", "product_id"),
    ("sales_orders_header", "so_id", "invoices", "so_id"),
    ("customers", "customer_id", "invoices", "customer_id"),
    ("branches", "branch_id", "invoices", "branch_id"),
    ("invoices", "invoice_id", "payments", "invoice_id"),
    ("products", "product_id", "stock_ledger", "product_id"),
    ("branches", "branch_id", "stock_ledger", "branch_id"),
    ("branches", "branch_id", "customers", "branch_id"),
]


# Task 4 part 1: candidate primary keys
PRIMARY_KEYS = {
    "branches": ["branch_id"],
    "products": ["product_id"],
    "inventory_master": ["product_id", "branch_id"],
    "suppliers": ["supplier_id"],
    "customers": ["customer_id"],
    "invoices": ["invoice_id"],
    "payments": ["payment_id"],
    "purchase_orders_header": ["po_id"],
    "purchase_orders_lines": ["po_id", "line_number"],
    "sales_orders_header": ["so_id"],
    "sales_orders_lines": ["so_id", "line_number"],
    "stock_ledger": ["movement_id"],
}

**Reading the output.** No output is expected. `FK_RELATIONSHIPS` holds the 17 candidate parent→child links and `PRIMARY_KEYS` the candidate key of each of the 12 tables. They are *candidates* inferred from column names; Task 4 tests them against the real records.

In [4]:
# Reuse the connection if this notebook already has one; otherwise open read-only.
try:
    con
except NameError:
    con = duckdb.connect(DB_PATH, read_only=True)

con.execute("SHOW TABLES").df()

,name
0,branches
1,customers
2,inventory_master
3,invoices
4,payments
5,products
6,purchase_orders_header
7,purchase_orders_lines
8,sales_orders_header
9,sales_orders_lines


**Reading the output.** A one-column list of the 12 tables: `branches`, `customers`, `inventory_master`, `invoices`, `payments`, `products`, `purchase_orders_header`, `purchase_orders_lines`, `sales_orders_header`, `sales_orders_lines`, `stock_ledger`, `suppliers`. If any is missing, the wrong folder or database file is open.

---
## 1. Task 1: Type casting and standardisation

**Objective.** Make every column usable for calculation: numbers stored as numbers, identifiers stored as text, composite text split into parts.

**Why it matters.** Capacity stored as `"45230 sqft"` cannot be summed or divided; `"40x25x25"` cannot give a volume; a postal code stored as a number can lose leading zeros. Space-utilisation and throughput KPIs in Week 6 depend on these conversions.

**Approach.**

| Column | Problem | Fix |
|---|---|---|
| `branches.warehouse_capacity` | Text with unit (`45230 sqft`) | New numeric column `warehouse_capacity_sqft`; original kept |
| `products.dimensions_cm` | Three measurements in one text field | New `length_cm`, `width_cm`, `height_cm` (regex), then `volume_m3` = L × W × H ÷ 1,000,000 |
| `customers.pincode`, `suppliers.pincode` | Identifier stored as a number | 6-character text |
| Date columns (7 tables) | Check type, format, missing values | Review only |
| Category columns (9 tables) | Check for stray spaces or spelling variants | Review only |

Each original column is preserved, so cleaned and raw values can always be compared.

In [5]:
def clean_branches(con) -> pd.DataFrame:
    """Add warehouse_capacity_sqft (numeric) and keep the original text column."""
    df = con.execute("SELECT * FROM branches").df()
    df["warehouse_capacity_sqft"] = (
        df["warehouse_capacity"]
        .str.replace("sqft", "", regex=False)
        .str.strip()
        .astype(int)
    )
    return df


def clean_products(con) -> pd.DataFrame:
    """Split dimensions_cm into length/width/height and add volume in m3."""
    df = con.execute("SELECT * FROM products").df()
    dims = df["dimensions_cm"].str.extract(
        r"(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*x\s*(\d+\.?\d*)"
    )
    dims.columns = ["length_cm", "width_cm", "height_cm"]
    df = pd.concat([df, dims.astype(float)], axis=1)
    df["volume_m3"] = df["length_cm"] * df["width_cm"] * df["height_cm"] / 1_000_000
    return df


def standardise_pincode(series: pd.Series) -> pd.Series:
    """Postal codes are identifiers, not numbers: store as 6-character text.
    (New in this file - recommended in the week 1 profiling, not yet applied.)"""
    return series.astype("Int64").astype(str).str.zfill(6)


def clean_customers(con) -> pd.DataFrame:
    df = con.execute("SELECT * FROM customers").df()
    df["pincode"] = standardise_pincode(df["pincode"])
    return df


def clean_suppliers(con) -> pd.DataFrame:
    df = con.execute("SELECT * FROM suppliers").df()
    df["pincode"] = standardise_pincode(df["pincode"])
    return df

**Reading the output.** No output: this cell only *defines* four cleaning functions. `clean_branches` and `clean_products` add the new numeric columns; `clean_customers` and `clean_suppliers` convert `pincode` to text; `standardise_pincode` is the shared helper. Defining them separately keeps the logic reusable (Task 5 moves it into `src/cleaning.py`).

In [6]:
def validate_task1_outputs(branches: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    """Final validation of the Task 1 transformations (one row per check)."""
    dims = ["length_cm", "width_cm", "height_cm"]
    rows = [
        ("capacity_sqft missing", int(branches["warehouse_capacity_sqft"].isna().sum())),
        ("capacity_sqft not positive", int((branches["warehouse_capacity_sqft"] <= 0).sum())),
        ("dimensions missing", int(products[dims].isna().sum().sum())),
        ("dimensions not positive", int((products[dims] <= 0).sum().sum())),
        ("volume_m3 missing", int(products["volume_m3"].isna().sum())),
    ]
    out = pd.DataFrame(rows, columns=["check", "exceptions"])
    out["status"] = out["exceptions"].map(lambda n: "PASSED" if n == 0 else "FLAGGED")
    return out


def date_column_check(con) -> pd.DataFrame:
    """Valid vs missing dates for every reviewed date column."""
    rows = []
    for table, cols in DATE_COLUMNS.items():
        for col in cols:
            total, valid = con.execute(
                f"SELECT COUNT(*), COUNT({col}) FROM {table}"
            ).fetchone()
            rows.append((table, col, total, valid, total - valid))
    return pd.DataFrame(rows, columns=["table_name", "column_name",
                                       "total_records", "valid_dates", "missing_dates"])


def po_status_vs_received_date(con) -> pd.DataFrame:
    """Explains the 2,370 missing received_date values."""
    return con.execute("""
        SELECT po_status,
               COUNT(*)                        AS total_orders,
               COUNT(received_date)            AS received_dates,
               COUNT(*) - COUNT(received_date) AS missing_dates
        FROM purchase_orders_header
        GROUP BY po_status
        ORDER BY total_orders DESC
    """).df()


def categorical_check(con) -> pd.DataFrame:
    """Distinct values with frequency; flags values that differ after trim/lower."""
    frames = []
    for table, cols in CATEGORICAL_COLUMNS.items():
        for col in cols:
            df = con.execute(
                f"SELECT {col} AS original_value, COUNT(*) AS frequency "
                f"FROM {table} GROUP BY {col}"
            ).df()
            df.insert(0, "column_name", col)
            df.insert(0, "table_name", table)
            frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out["cleaned_value"] = out["original_value"].astype(str).str.strip()
    out["standardised_candidate"] = out["cleaned_value"].str.lower()
    out["needs_cleaning"] = out["original_value"].astype(str) != out["cleaned_value"]
    return out


def categorical_inconsistencies(check: pd.DataFrame) -> pd.DataFrame:
    """Columns where two spellings collapse into one value after trim + lower."""
    g = (check.groupby(["table_name", "column_name", "standardised_candidate"])
              ["original_value"].nunique().reset_index(name="spellings"))
    return g[g["spellings"] > 1]

**Reading the output.** No output: these are the **validation** and **review** functions: a checker for the Task 1 transformations, a date-column profiler, an explainer for missing `received_date` values, and a category spell-checker (distinct values with frequencies; flags any value that changes after trimming, and any two spellings that collapse to one after trim + lower-casing).

In [7]:
branches_clean = clean_branches(con)
products_clean = clean_products(con)

display(branches_clean[["branch_name", "warehouse_capacity", "warehouse_capacity_sqft"]])
display(products_clean[["product_name", "dimensions_cm", "length_cm", "width_cm",
                        "height_cm", "volume_m3"]].head(10))

,branch_name,warehouse_capacity,warehouse_capacity_sqft
0,Delhi Central,45230 sqft,45230
1,Pune Distribution,38750 sqft,38750
2,Chennai South Hub,41890 sqft,41890
3,Hyderabad Logistics,36210 sqft,36210
4,Kolkata East Depot,29880 sqft,29880
5,Ahmedabad West Hub,34120 sqft,34120


,product_name,dimensions_cm,length_cm,width_cm,height_cm,volume_m3
0,Hydraulic Pump HP-300,40x25x25,40.0,25.0,25.0,0.025000
1,Engine Oil Filter OF-90,10x8x8,10.0,8.0,8.0,0.000640
2,Air Filter AF-120,18x12x12,18.0,12.0,12.0,0.002592
3,Track Chain TC-45,180x40x35,180.0,40.0,35.0,0.252000
4,Fuel Injector FI-220,20x12x12,20.0,12.0,12.0,0.002880
5,Alternator ALT-500,30x20x20,30.0,20.0,20.0,0.012000
6,Hydraulic Hose HH-60,90x4x4,90.0,4.0,4.0,0.001440
7,Bucket Tooth BT-10,12x6x4,12.0,6.0,4.0,0.000288
8,Starter Motor SM-700,32x18x18,32.0,18.0,18.0,0.010368
9,Turbocharger TB-900,40x25x25,40.0,25.0,25.0,0.025000


**Reading the output.** Two tables.
* **Branches (6 rows):** the text capacity beside the numeric value, e.g. Delhi Central `45230 sqft` → `45230`. The six values range from 29,880 sqft (Kolkata East Depot) to 45,230 sqft (Delhi Central).
* **Products (first 10 of 30):** each `dimensions_cm` string beside its parts and volume, e.g. Hydraulic Pump HP-300 `40x25x25` → 40 × 25 × 25 cm → **0.025 m³**; Track Chain TC-45 `180x40x35` → **0.252 m³**, the bulkiest of the ten shown.

Check the arithmetic on one row yourself: 40 × 25 × 25 = 25,000 cm³ ÷ 1,000,000 = 0.025 m³.

In [8]:
task1_validation = validate_task1_outputs(branches_clean, products_clean)
task1_validation

,check,exceptions,status
0,capacity_sqft missing,0,PASSED
1,capacity_sqft not positive,0,PASSED
2,dimensions missing,0,PASSED
3,dimensions not positive,0,PASSED
4,volume_m3 missing,0,PASSED


**Reading the output.** Five checks (missing or non-positive capacity, missing or non-positive dimensions, missing volume), each with a count of exceptions and a status. **All five should show 0 exceptions and `PASSED`.** A zero-volume or negative dimension would break every space calculation downstream, so this is the gate before moving on.

In [9]:
date_check = date_column_check(con)
date_check

,table_name,column_name,total_records,valid_dates,missing_dates
0,products,last_purchase_date,30,30,0
1,customers,customer_since,500,500,0
2,customers,last_purchase_date,500,500,0
3,invoices,invoice_date,18033,18033,0
4,invoices,due_date,18033,18033,0
5,payments,payment_date,19257,19257,0
6,purchase_orders_header,order_date,24000,24000,0
7,purchase_orders_header,expected_delivery_date,24000,24000,0
8,purchase_orders_header,received_date,24000,21630,2370
9,sales_orders_header,order_date,20000,20000,0


**Reading the output.** One row per reviewed date column (12 rows): total records, valid dates and missing dates. Every column should be fully populated **except `purchase_orders_header.received_date`**, which has **2,370 missing** (21,630 of 24,000 valid). Because the columns are already stored as `DATE`, no type conversion was needed. The missing values are explained in the next cell.

In [10]:
# Explains the missing received_date values
po_status_vs_received_date(con)

,po_status,total_orders,received_dates,missing_dates
0,Received,21630,21630,0
1,Cancelled,2370,0,2370


**Reading the output.** Missing `received_date` values split by purchase-order status:

| po_status | total | with received date | missing |
|---|---:|---:|---:|
| Received | 21,630 | 21,630 | 0 |
| Cancelled | 2,370 | 0 | 2,370 |

Every missing date belongs to a **cancelled** order, and every received order has a date. A cancelled order is never received, so these are valid business blanks, **not errors**. No date was invented to fill them.

In [11]:
cat_check = categorical_check(con)
print("Values needing trim:", int(cat_check["needs_cleaning"].sum()))
categorical_inconsistencies(cat_check)     # empty = no spelling clashes

Values needing trim: 0


,table_name,column_name,standardised_candidate,spellings


**Reading the output.** The first line counts category values that contain leading or trailing spaces. The table below it lists any column where two spellings collapse into one after trimming and lower-casing (for example `Cash` and `cash `). **An empty table (and a count of 0) means every category value is already clean** and needs no standardisation.

### Task 1: Summary

| Item | Result | Decision |
|---|---|---|
| Warehouse capacity | Text → numeric sqft for all 6 branches | **FIX** (original kept) |
| Product dimensions and volume | Parsed for all 30 products; volume in m³ | **FIX** (original kept) |
| Postal codes | Converted from number to 6-character text | **FIX** |
| Date columns | All already `DATE`; only `received_date` has blanks | **ACCEPT** |
| 2,370 missing `received_date` | All belong to cancelled POs | **ACCEPT** (no dates invented) |
| Category labels | No spacing or spelling clashes | **ACCEPT** (no change) |

**Status: COMPLETED.** The raw database was not touched; every change lives in separate DataFrames and, from Task 5, in Parquet files.

---
## 2. Task 2: Stock position reconstruction and reconciliation

**Objective.** Rebuild each product's stock at each branch from the movement history and compare it with two recorded figures: the ledger's own `running_balance` and `inventory_master.current_stock`.

**Why it matters.** Inventory health, turnover, ageing and optimisation (Weeks 5 and 10) all start from "how much stock is there". If the ledger cannot be trusted, none of those numbers can.

**The idea.**

> Opening stock + IN − OUT ± ADJUSTMENT = closing stock

**Two difficulties found in this data.**

1. **ADJUSTMENT has no sign.** All 2,454 adjustment quantities are positive, so the data does not say whether each one added or removed stock.
2. **Row order is not reliable.** Many movements share a date and there is no sequence column. Sorting by `movement_id` sorts *as text*, so `MOV-IN-148015` comes before `MOV-IN-77415`. A running-balance check that depends on row order therefore produces false mismatches. An earlier order-dependent check reported **179 of 180 pairs mismatching**; that figure was caused by ordering and unsigned adjustments, not by bad data.

**Approach.** Use a test that does **not** depend on order. For each product-branch pair, take the expected balance before adjustments, then ask whether the recorded balance can be reached by giving each adjustment a plus or minus sign. A balance is *plausible* when its distance from the expected figure is no larger than the total adjustment quantity and has the same odd/even parity.

> **Limitation to remember.** This is a *necessary* condition, so it can rule a balance out but cannot strictly prove it. With 10–19 adjustments per pair there is a lot of flexibility, so passing is strong evidence of consistency, not a proof. Per-row adjustment direction stays unresolved.

In [12]:
def stock_movement_summary(con) -> pd.DataFrame:
    """Movement types, counts and quantity signs (task 2.1)."""
    return con.execute("""
        SELECT movement_type,
               COUNT(*)      AS total_records,
               SUM(quantity) AS total_quantity,
               MIN(quantity) AS minimum_quantity,
               MAX(quantity) AS maximum_quantity,
               SUM(CASE WHEN quantity > 0 THEN 1 ELSE 0 END) AS positive_movements,
               SUM(CASE WHEN quantity < 0 THEN 1 ELSE 0 END) AS negative_movements,
               SUM(CASE WHEN quantity = 0 THEN 1 ELSE 0 END) AS zero_movements
        FROM stock_ledger
        GROUP BY movement_type
        ORDER BY movement_type
    """).df()


def stock_reconstruction(con) -> pd.DataFrame:
    """Opening stock + IN - OUT per product-branch; adjustments kept apart."""
    return con.execute("""
        WITH movement_totals AS (
            SELECT product_id, branch_id,
                   SUM(CASE WHEN movement_type = 'IN'  THEN quantity ELSE 0 END) AS total_in,
                   SUM(CASE WHEN movement_type = 'OUT' THEN quantity ELSE 0 END) AS total_out,
                   SUM(CASE WHEN movement_type = 'ADJUSTMENT' THEN quantity ELSE 0 END) AS total_adjustment,
                   SUM(CASE WHEN movement_type = 'ADJUSTMENT' THEN 1 ELSE 0 END) AS adjustment_records
            FROM stock_ledger
            GROUP BY product_id, branch_id
        )
        SELECT i.product_id, i.branch_id, i.opening_stock, i.current_stock,
               COALESCE(m.total_in, 0)           AS total_in,
               COALESCE(m.total_out, 0)          AS total_out,
               COALESCE(m.total_adjustment, 0)   AS total_adjustment,
               COALESCE(m.adjustment_records, 0) AS adjustment_records,
               i.opening_stock + COALESCE(m.total_in, 0) - COALESCE(m.total_out, 0)
                   AS expected_before_adjustments
        FROM inventory_master i
        LEFT JOIN movement_totals m
               ON i.product_id = m.product_id AND i.branch_id = m.branch_id
        ORDER BY i.product_id, i.branch_id
    """).df()


def last_day_balances(con) -> pd.DataFrame:
    """Every running_balance on each product-branch's last date.
    Several movements can share that date and their true order is not stored,
    so all of them are kept instead of guessing one."""
    return con.execute("""
        SELECT s.product_id, s.branch_id, s.movement_date AS last_date,
               s.movement_id, s.running_balance
        FROM stock_ledger s
        JOIN (SELECT product_id, branch_id, MAX(movement_date) AS last_date
              FROM stock_ledger GROUP BY product_id, branch_id) l
          ON s.product_id = l.product_id
         AND s.branch_id  = l.branch_id
         AND s.movement_date = l.last_date
    """).df()

**Reading the output.** No output: three functions are defined. `stock_movement_summary` profiles the movement types; `stock_reconstruction` builds opening + IN − OUT per product-branch (adjustments kept in a separate column); `last_day_balances` returns **every** running balance recorded on each pair's last date, because the true last movement of the day cannot be identified.

In [13]:
def _is_plausible(net: pd.Series, total_adj: pd.Series) -> pd.Series:
    """A signed sum of positive quantities: |net| <= total and same parity."""
    return (net.abs() <= total_adj) & ((total_adj - net) % 2 == 0)


def stock_adjustment_test(con) -> pd.DataFrame:
    """Order-independent test: opening + IN - OUT + (signed adjustments) = balance.

    plausible_ledger  : final ledger balance is reachable with signed adjustments
    plausible_current : inventory_master.current_stock is reachable the same way
    """
    recon = stock_reconstruction(con)
    last = last_day_balances(con)

    cand = last.merge(
        recon[["product_id", "branch_id", "expected_before_adjustments", "total_adjustment"]],
        on=["product_id", "branch_id"], how="left")
    cand["net_adjustment"] = cand["running_balance"] - cand["expected_before_adjustments"]
    cand["ok"] = _is_plausible(cand["net_adjustment"], cand["total_adjustment"])
    cand["_id_num"] = pd.to_numeric(cand["movement_id"].str.extract(r"(\d+)$")[0], errors="coerce")
    cand = cand.sort_values(["product_id", "branch_id", "ok", "_id_num"],
                            ascending=[True, True, False, False])
    best = cand.groupby(["product_id", "branch_id"], as_index=False).first()
    rows_per_pair = (last.groupby(["product_id", "branch_id"]).size()
                         .rename("last_day_rows").reset_index())

    out = (recon.merge(best[["product_id", "branch_id", "last_date", "running_balance",
                             "net_adjustment", "ok"]],
                       on=["product_id", "branch_id"], how="left")
                .merge(rows_per_pair, on=["product_id", "branch_id"], how="left"))
    out = out.rename(columns={"running_balance": "final_ledger_balance",
                              "ok": "plausible_ledger"})
    out["net_adj_vs_current"] = out["current_stock"] - out["expected_before_adjustments"]
    out["plausible_current"] = _is_plausible(out["net_adj_vs_current"], out["total_adjustment"])
    out["ledger_equals_current"] = out["final_ledger_balance"] == out["current_stock"]
    return out


def summarise_stock_test(result: pd.DataFrame) -> pd.DataFrame:
    n = len(result)
    s = pd.DataFrame({
        "check": ["Final ledger balance reachable with signed adjustments",
                  "inventory_master.current_stock reachable with signed adjustments",
                  "Final ledger balance equals current_stock",
                  "Pairs with several movements on the last date (order ambiguous)"],
        "pass_count": [int(result["plausible_ledger"].sum()),
                       int(result["plausible_current"].sum()),
                       int(result["ledger_equals_current"].sum()),
                       int((result["last_day_rows"] > 1).sum())],
        "total": [n, n, n, n],
    })
    s["pass_pct"] = (s["pass_count"] / s["total"] * 100).round(1)
    return s

**Reading the output.** No output: these functions implement the order-independent test. `_is_plausible` applies the "distance ≤ total adjustments, same parity" rule. `stock_adjustment_test` applies it to the final ledger balance and to `current_stock` for all 180 pairs, trying each last-day balance and keeping the one that fits. `summarise_stock_test` turns the result into a four-line scorecard.

In [14]:
stock_movement_summary(con)

,movement_type,total_records,total_quantity,minimum_quantity,maximum_quantity,positive_movements,negative_movements,zero_movements
0,ADJUSTMENT,2454,66990.0,5,50,2454.0,0.0,0.0
1,IN,127611,20397795.0,20,300,127611.0,0.0,0.0
2,OUT,107165,1124832.0,1,20,107165.0,0.0,0.0


**Reading the output.** One row per movement type.

| Type | Records | Total quantity | Quantity range |
|---|---:|---:|---|
| ADJUSTMENT | 2,454 | 66,990 | 5 – 50 |
| IN | 127,611 | 20,397,795 | 20 – 300 |
| OUT | 107,165 | 1,124,832 | 1 – 20 |

Three things matter. **(1)** Every quantity is positive (`negative_movements` and `zero_movements` are 0), so direction comes from the movement *type*, and for ADJUSTMENT it is unknown. **(2)** Stock coming in (20.4 M units) is about **18 times** stock going out (1.12 M units). This is the root of the overstock finding in Week 5. **(3)** The OUT quantities are small and the IN quantities large, so balances grow steadily.

In [15]:
stock_test = stock_adjustment_test(con)
summarise_stock_test(stock_test)

,check,pass_count,total,pass_pct
0,Final ledger balance reachable with signed adj...,180,180,100.0
1,inventory_master.current_stock reachable with ...,171,180,95.0
2,Final ledger balance equals current_stock,166,180,92.2
3,Pairs with several movements on the last date ...,33,180,18.3


**Reading the output.** A four-line scorecard over the 180 product-branch pairs:

| Check | Pass | Meaning |
|---|---:|---|
| Final ledger balance reachable with signed adjustments | **180 / 180** | The ledger is internally consistent |
| `current_stock` reachable with signed adjustments | **171 / 180** | 9 pairs cannot be reconciled by any sign pattern |
| Final ledger balance equals `current_stock` | **166 / 180** | 14 pairs differ |
| Pairs with several movements on the last date | **33** | Their "latest" balance is ambiguous |

The ledger passes everywhere. The remaining differences are all on the `inventory_master.current_stock` side.

In [16]:
# Pairs failing either test (empty = ledger is consistent)
stock_test[~stock_test["plausible_ledger"] | ~stock_test["plausible_current"]][[
    "product_id", "branch_id", "expected_before_adjustments", "final_ledger_balance",
    "current_stock", "total_adjustment", "net_adjustment", "net_adj_vs_current"]]

,product_id,branch_id,expected_before_adjustments,final_ledger_balance,current_stock,total_adjustment,net_adjustment,net_adj_vs_current
7,P002,CHN001,106204.0,106182,105885,326.0,-22.0,-319.0
23,P004,PUN001,104925.0,105024,104859,605.0,99.0,-66.0
54,P010,AHM001,115504.0,115549,115149,211.0,45.0,-355.0
122,P021,DEL001,116235.0,116107,115902,494.0,-128.0,-333.0
135,P023,HYD001,106109.0,105981,105798,252.0,-128.0,-311.0
147,P025,HYD001,114257.0,114128,113613,207.0,-129.0,-644.0
155,P026,PUN001,105139.0,105335,105190,576.0,196.0,51.0
164,P028,DEL001,106838.0,106967,106744,675.0,129.0,-94.0
172,P029,KOL001,105374.0,105249,105010,273.0,-125.0,-364.0


**Reading the output.** The product-branch pairs that fail either test. It lists `expected_before_adjustments`, the final ledger balance, `current_stock`, total adjustments and the net adjustment each would need. Expect no row to fail the **ledger** test and a short list (about 9 pairs) to fail the **`current_stock`** test. An empty table would mean everything reconciles. These rows are the items to investigate, and each is later carried into the issue log.

### 2.4 Follow-up: where `current_stock` differs from the ledger

For the pairs where `current_stock` does not equal the final ledger balance, we ask a sharper question: does `current_stock` equal **any** balance recorded on the last date? If yes, the difference is only about *which movement came last* (an ordering ambiguity), not bad data. If no, the difference is **genuine** and stays flagged.

In [17]:
mismatch = stock_test[~stock_test["ledger_equals_current"]].copy()
mismatch["current_minus_ledger"] = mismatch["current_stock"] - mismatch["final_ledger_balance"]

# If current_stock equals ANY balance recorded on the last date, the difference is
# only about which movement came last, not a data error.
last_day = last_day_balances(con)[["product_id", "branch_id", "running_balance"]]
hit = mismatch.merge(last_day, on=["product_id", "branch_id"], how="left")
hit["hit"] = hit["running_balance"] == hit["current_stock"]
explained = (hit.groupby(["product_id", "branch_id"])["hit"].any()
                .reset_index(name="explained_by_ordering"))

mismatch = mismatch.merge(explained, on=["product_id", "branch_id"], how="left")
mismatch["explained_by_ordering"] = mismatch["explained_by_ordering"].fillna(False).astype(bool)

print("current_stock differs from ledger:", len(mismatch))
print("  explained by last-day ordering :", int(mismatch["explained_by_ordering"].sum()))
print("  genuine differences            :", int((~mismatch["explained_by_ordering"]).sum()))

mismatch[["product_id", "branch_id", "final_ledger_balance", "current_stock",
          "current_minus_ledger", "plausible_current", "last_day_rows",
          "explained_by_ordering"]]

current_stock differs from ledger: 14
  explained by last-day ordering : 14
  genuine differences            : 0


,product_id,branch_id,final_ledger_balance,current_stock,current_minus_ledger,plausible_current,last_day_rows,explained_by_ordering
0,P002,CHN001,106182,105885,-297,False,2,True
1,P004,PUN001,105024,104859,-165,False,2,True
2,P008,AHM001,109317,109281,-36,True,2,True
3,P010,AHM001,115549,115149,-400,False,3,True
4,P021,DEL001,116107,115902,-205,False,2,True
5,P023,HYD001,105981,105798,-183,False,2,True
6,P025,HYD001,114128,113613,-515,False,3,True
7,P026,HYD001,108546,108348,-198,True,2,True
8,P026,PUN001,105335,105190,-145,False,2,True
9,P028,DEL001,106967,106744,-223,False,3,True


**Reading the output.** Three counts, then a table. The counts split the pairs where `current_stock` differs from the ledger (14) into **explained by last-day ordering** (harmless: `current_stock` matches one of the last-day balances) and **genuine differences** (no last-day balance matches). The table lists each pair with its ledger balance, `current_stock`, the gap, and the explanation flag. Genuine rows are the real stock discrepancies and are flagged in the issue log; they are not silently corrected.

### Task 2: Summary

| Question | Answer |
|---|---|
| Can stock be rebuilt from the ledger? | Yes: opening + IN − OUT gives an expected balance for all 180 pairs |
| Is the ledger internally consistent? | Yes: 180 / 180 final balances are consistent with signed adjustments |
| Does `current_stock` agree? | 166 / 180 equal the ledger balance; 171 / 180 are consistent with it; the rest are listed above |
| Are adjustment directions known? | **No.** All 2,454 are positive numbers; direction is inferred only in aggregate |
| Why did the first attempt show 179 mismatches? | Text-sorted `movement_id` and unsigned adjustments, not bad data |

**Decisions.** **ACCEPT** the ledger as the stock source of truth. **FLAG** the adjustment-direction gap, the 33 ambiguous last-day pairs and the genuine `current_stock` differences. The adjustment direction needs a business answer (see the Week 1 request for *Fields Documentation*) before the Week 11 shrinkage analysis.

**Status: COMPLETED; DISCREPANCIES FLAGGED.**

---
## 3. Task 3: Financial reconciliation

**Objective.** Confirm that the money fields agree with each other at every level: line → order → invoice → payment → customer.

**Why it matters.** Revenue, margin, receivables, DSO and customer value (Weeks 3, 8 and 12) are all sums of these fields. If they do not reconcile, every financial KPI inherits the error.

**Checks, in order.**

| # | Check | Rule |
|---|---|---|
| 3.1 | Line arithmetic | `line_total = quantity × price`; `GST = line_total × rate ÷ 100`; `line_grand_total = line_total + GST`; GST rate is 18 or 28 |
| 3.2 | Header vs lines | Header base, GST and grand total equal the sum of their lines |
| 3.3 | Invoices vs payments | Payments summed per invoice and compared with the invoice grand total |
| 3.4 | Customer purchase value | Stored `total_purchase_value` equals the value recomputed from delivered orders |

All comparisons use a 0.01-rupee tolerance. Purchase orders get the same line and header checks as sales orders.

**A design point for 3.3.** `invoice_id` is reused by different invoices (found in Task 4). A naive join of invoices to payments on `invoice_id` would attach the *same* payments to *each* duplicate, double-counting money and making unpaid invoices look paid. The reconciliation here sums payments per ID first and labels invoices with a reused ID **Ambiguous** instead of guessing.

In [18]:
_LINE_CONFIG = {
    "sales": dict(lines="sales_orders_lines", price="unit_price"),
    "purchase": dict(lines="purchase_orders_lines", price="unit_cost"),
}


_HEADER_CONFIG = {
    "sales": dict(header="sales_orders_header", lines="sales_orders_lines", key="so_id",
                  h_base="total_order_value", h_gst="total_gst_amount", h_grand="grand_total",
                  l_base="line_total", l_gst="gst_amount", l_grand="line_grand_total"),
    "purchase": dict(header="purchase_orders_header", lines="purchase_orders_lines", key="po_id",
                     h_base="total_cost", h_gst="total_gst_amount", h_grand="grand_total",
                     l_base="line_total", l_gst="gst_amount", l_grand="line_grand_total"),
}


def line_arithmetic_check(con, kind: str = "sales", tol: float = MONEY_TOLERANCE) -> pd.DataFrame:
    """One-row summary: line_total = qty x price, GST = line_total x rate,
    line_grand_total = line_total + GST, and the GST rates found."""
    c = _LINE_CONFIG[kind]
    row = con.execute(f"""
        SELECT COUNT(*) AS total_lines,
          SUM(CASE WHEN ABS(line_total - quantity * {c['price']}) >= {tol} THEN 1 ELSE 0 END) AS line_total_mismatch,
          SUM(CASE WHEN ABS(gst_amount - line_total * gst_rate / 100.0) >= {tol} THEN 1 ELSE 0 END) AS gst_mismatch,
          SUM(CASE WHEN ABS(line_grand_total - (line_total + gst_amount)) >= {tol} THEN 1 ELSE 0 END) AS grand_total_mismatch,
          SUM(CASE WHEN gst_rate NOT IN (18, 28) THEN 1 ELSE 0 END) AS unexpected_gst_rate
        FROM {c['lines']}
    """).df()
    row.insert(0, "table", c["lines"])
    return row


def header_vs_lines(con, kind: str = "sales", tol: float = MONEY_TOLERANCE) -> pd.DataFrame:
    """Header totals vs the sum of their lines (base, GST, grand total)."""
    c = _HEADER_CONFIG[kind]
    df = con.execute(f"""
        WITH l AS (
            SELECT {c['key']}, COUNT(*) AS line_count,
                   SUM({c['l_base']}) AS lines_base,
                   SUM({c['l_gst']}) AS lines_gst,
                   SUM({c['l_grand']}) AS lines_grand
            FROM {c['lines']} GROUP BY {c['key']}
        )
        SELECT h.{c['key']} AS id,
               h.{c['h_base']}  AS header_base,  l.lines_base,
               h.{c['h_gst']}   AS header_gst,   l.lines_gst,
               h.{c['h_grand']} AS header_grand, l.lines_grand,
               COALESCE(l.line_count, 0) AS line_count
        FROM {c['header']} h
        LEFT JOIN l ON h.{c['key']} = l.{c['key']}
    """).df()
    df["diff_base"] = df["header_base"] - df["lines_base"]
    df["diff_gst"] = df["header_gst"] - df["lines_gst"]
    df["diff_grand"] = df["header_grand"] - df["lines_grand"]
    bad = ((df[["diff_base", "diff_gst", "diff_grand"]].abs() >= tol).any(axis=1)
           | df["lines_grand"].isna())
    df["status"] = bad.map({True: "Mismatching", False: "Matching"})
    return df

**Reading the output.** No output: two configuration dictionaries and two functions. The dictionaries map "sales" and "purchase" to their table and column names so the **same** code checks both. `line_arithmetic_check` returns a one-row summary of line-level exceptions; `header_vs_lines` compares each header with the sum of its lines and labels it `Matching` or `Mismatching` (a header with no lines also counts as a mismatch).

In [19]:
def invoice_payment_reconciliation(con, tol: float = MONEY_TOLERANCE) -> pd.DataFrame:
    """Invoice vs payments, built so duplicated invoice_ids cannot double-count.

    Payments are summed per invoice_id first. Invoices whose invoice_id appears
    more than once are labelled 'Ambiguous' because their payments cannot be
    assigned to one invoice.
    """
    df = con.execute("""
        WITH paid AS (
            SELECT invoice_id, SUM(payment_amount) AS total_paid, COUNT(*) AS payment_count
            FROM payments GROUP BY invoice_id
        ), dup AS (
            SELECT invoice_id, COUNT(*) AS invoice_rows FROM invoices GROUP BY invoice_id
        )
        SELECT i.invoice_id, i.so_id, i.payment_status, i.grand_total,
               COALESCE(p.total_paid, 0)    AS total_paid,
               COALESCE(p.payment_count, 0) AS payment_count,
               d.invoice_rows
        FROM invoices i
        LEFT JOIN paid p ON i.invoice_id = p.invoice_id
        JOIN dup d       ON i.invoice_id = d.invoice_id
    """).df()
    df["outstanding_balance"] = df["grand_total"] - df["total_paid"]

    def classify(r):
        if r["invoice_rows"] > 1:
            return "Ambiguous (duplicate invoice_id)"
        if r["total_paid"] <= tol:
            return "Unpaid"
        if r["outstanding_balance"] > tol:
            return "Partially paid"
        if r["outstanding_balance"] < -tol:
            return "Overpaid"
        return "Fully paid"

    df["payment_result"] = df.apply(classify, axis=1)
    return df


def payment_status_crosstab(recon: pd.DataFrame) -> pd.DataFrame:
    """Stored payment_status vs the status derived from payments."""
    return pd.crosstab(recon["payment_status"], recon["payment_result"], margins=True)


def customer_purchase_reconciliation(con, tol: float = MONEY_TOLERANCE) -> pd.DataFrame:
    """Stored customers.total_purchase_value vs recomputed from sales orders.
    Cancelled orders are excluded; both with-GST and before-GST are shown."""
    df = con.execute("""
        WITH calc AS (
            SELECT customer_id,
                   COUNT(*)               AS delivered_orders,
                   SUM(total_order_value) AS delivered_base,
                   SUM(grand_total)       AS delivered_grand
            FROM sales_orders_header
            WHERE LOWER(TRIM(order_status)) = 'delivered'
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.total_purchase_value AS stored_value,
               COALESCE(k.delivered_orders, 0) AS delivered_orders,
               COALESCE(k.delivered_base, 0)   AS delivered_base,
               COALESCE(k.delivered_grand, 0)  AS delivered_grand
        FROM customers c
        LEFT JOIN calc k ON c.customer_id = k.customer_id
    """).df()
    df["diff_vs_grand"] = df["stored_value"] - df["delivered_grand"]
    df["diff_vs_base"] = df["stored_value"] - df["delivered_base"]
    df["status"] = df["diff_vs_grand"].abs().map(lambda x: "Matching" if x < tol else "Mismatching")
    return df


def summarise_customer_reconciliation(df: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "customers": len(df),
        "matching": int((df["status"] == "Matching").sum()),
        "mismatching": int((df["status"] == "Mismatching").sum()),
        "stored_as_pct_of_recomputed": round(
            df["stored_value"].sum() / df["delivered_grand"].sum() * 100, 2),
        "correlation": round(df["stored_value"].corr(df["delivered_grand"]), 3),
    })

**Reading the output.** No output: three functions. `invoice_payment_reconciliation` builds the safe invoice-vs-payment table with the labels *Unpaid, Partially paid, Fully paid, Overpaid, Ambiguous (duplicate invoice_id)*. `payment_status_crosstab` compares the **stored** `payment_status` with the **derived** result. `customer_purchase_reconciliation` recomputes each customer's purchases from **delivered** orders only (cancelled orders are not purchases) and shows both the with-GST and before-GST totals; `summarise_customer_reconciliation` condenses it into one summary.

In [20]:
line_checks = pd.concat([line_arithmetic_check(con, "sales"),
                         line_arithmetic_check(con, "purchase")], ignore_index=True)
line_checks

,table,total_lines,line_total_mismatch,gst_mismatch,grand_total_mismatch,unexpected_gst_rate
0,sales_orders_lines,130402,0.0,0.0,0.0,0.0
1,purchase_orders_lines,155495,0.0,0.0,0.0,0.0


**Reading the output.** Two rows, sales lines (130,402) and purchase lines (155,495), with counts of lines failing each rule. **All four exception columns should be 0**, meaning every line total equals quantity × price, GST is exactly the stated rate, line grand totals equal line total + GST and only the 18% and 28% GST rates appear. Any non-zero number is an exception to investigate.

In [21]:
sales_hv = header_vs_lines(con, "sales")
purchase_hv = header_vs_lines(con, "purchase")
print("Sales   :", sales_hv["status"].value_counts().to_dict())
print("Purchase:", purchase_hv["status"].value_counts().to_dict())

Sales   : {'Matching': 20000}
Purchase: {'Matching': 24000}


**Reading the output.** Two lines counting orders whose header agrees with the sum of their lines: sales `{'Matching': 20000}` and purchase `{'Matching': 24000}`. Any `Mismatching` entry would mean a header total that its lines do not support. Together with the previous cell this shows the order-level money is internally consistent.

In [22]:
invoice_payment = invoice_payment_reconciliation(con)
invoice_payment["payment_result"].value_counts()

payment_result
Fully paid                          12145
Partially paid                       3645
Unpaid                               1798
Ambiguous (duplicate invoice_id)      393
Overpaid                               52
Name: count, dtype: int64

**Reading the output.** A count of invoices by payment result (18,033 in total):

| Result | Invoices |
|---|---:|
| Fully paid | 12,145 |
| Partially paid | 3,645 |
| Unpaid | 1,798 |
| **Ambiguous (duplicate invoice_id)** | **393** |
| Overpaid | 52 |

The 393 ambiguous invoices cannot be reconciled until the duplicate IDs are resolved. The other 17,640 can.

In [23]:
# Stored payment_status vs status derived from payments
payment_status_crosstab(invoice_payment)

payment_result,Ambiguous (duplicate invoice_id),Fully paid,Overpaid,Partially paid,Unpaid,All
payment_status,,,,,,
Paid,272,12145,52,67,0,12536
Partially Paid,82,0,0,3578,0,3660
Unpaid,39,0,0,0,1798,1837
All,393,12145,52,3645,1798,18033


**Reading the output.** A cross-table: rows are the **stored** `payment_status` (Paid / Partially Paid / Unpaid), columns are the status **derived** from the payments. Cells on the natural diagonal are agreement. Outside the ambiguous group, stored `Unpaid` and `Partially Paid` agree with the payments every time. The exceptions are **67 invoices stored as `Paid` that were only partly paid**, so the stored status cannot be trusted for those and the status must be **derived from payments** instead.

In [24]:
customer_recon = customer_purchase_reconciliation(con)
summarise_customer_reconciliation(customer_recon)

customers                      500.000
matching                         0.000
mismatching                    500.000
stored_as_pct_of_recomputed      4.240
correlation                     -0.077
dtype: float64

**Reading the output.** A one-column summary: number of customers (500), how many match or mismatch, stored purchase value as a percentage of the recomputed value, and the correlation between them. The Week 1 profile found the stored value to be only about **5% of actual** with a correlation near **−0.08**, so expect almost every customer to mismatch. A value that is both far too small *and* uncorrelated means the stored field is not a usable summary of purchases and should be recomputed from transactions.

### 3.5 Follow-up: invoice and payment keys

Before concluding, the key structure is checked directly, because the duplicate-ID finding changes how invoices and payments may be joined.

In [25]:
inv_keys = con.execute("""
    SELECT COUNT(*)                   AS invoice_rows,
           COUNT(DISTINCT invoice_id) AS distinct_invoice_ids,
           COUNT(DISTINCT so_id)      AS distinct_so_ids
    FROM invoices
""").df()

pay_keys = con.execute("""
    SELECT COUNT(*)                   AS payment_rows,
           COUNT(DISTINCT payment_id) AS distinct_payment_ids,
           (SELECT COUNT(*) FROM (SELECT DISTINCT payment_id, invoice_id FROM payments))
                                      AS distinct_payment_invoice_pairs
    FROM payments
""").df()

display(inv_keys)
display(pay_keys)
print("so_id can identify an invoice:", inv_keys["distinct_so_ids"].iloc[0] == inv_keys["invoice_rows"].iloc[0])
print("(payment_id, invoice_id) is unique:",
      pay_keys["distinct_payment_invoice_pairs"].iloc[0] == pay_keys["payment_rows"].iloc[0])

,invoice_rows,distinct_invoice_ids,distinct_so_ids
0,18033,17836,18033


,payment_rows,distinct_payment_ids,distinct_payment_invoice_pairs
0,19257,19055,19257


so_id can identify an invoice: True
(payment_id, invoice_id) is unique: True


**Reading the output.** Two small tables and two `True/False` lines.
* **Invoices:** 18,033 rows but only about 17,836 distinct `invoice_id` values (197 fewer), while `so_id` should be distinct on every row (the first `True/False` line should read `True`). So **`so_id` can safely identify an invoice; `invoice_id` cannot.**
* **Payments:** 19,257 rows, about 19,055 distinct `payment_id` values (202 fewer). The pair `(payment_id, invoice_id)` should be unique (`True`), so the pair can serve as a payment key.

These two facts decide how later weeks must join the finance tables.

In [26]:
# Stored payment_status vs payments (ambiguous invoices excluded)
expected = {"Paid": ["Fully paid", "Overpaid"],
            "Partially Paid": ["Partially paid"],
            "Unpaid": ["Unpaid"]}

clear = invoice_payment[~invoice_payment["payment_result"].str.startswith("Ambiguous")].copy()
clear["consistent"] = [r in expected.get(s, []) for s, r in
                       zip(clear["payment_status"], clear["payment_result"])]

contradictions = clear[~clear["consistent"]]
print("Clear invoices:", len(clear), "| contradicting stored status:", len(contradictions))
display(contradictions.groupby(["payment_status", "payment_result"]).size().rename("invoices").reset_index())

contradictions.assign(pct_paid=(contradictions["total_paid"] / contradictions["grand_total"] * 100).round(1)
                      )["pct_paid"].describe().round(1)

Clear invoices: 17640 | contradicting stored status: 67


,payment_status,payment_result,invoices
0,Paid,Partially paid,67


count     67.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: pct_paid, dtype: float64

**Reading the output.** Of the **17,640** unambiguous invoices, how many have a stored status that **contradicts** the payments: **67**, all stored `Paid` but only partly paid. The small table groups them by (stored, derived) status; the last block summarises what percentage of the invoice had actually been paid. These 67 are flagged, and their status is derived from payments in the processed layer.

In [27]:
# Size of the overpayments
over = invoice_payment[invoice_payment["payment_result"] == "Overpaid"]
overpaid_amount = (-over["outstanding_balance"])
print("Overpaid invoices:", len(over), "| total overpaid (INR):", round(overpaid_amount.sum(), 2))
overpaid_amount.describe().round(2)

Overpaid invoices: 52 | total overpaid (INR): 0.52


count    52.00
mean      0.01
std       0.00
min       0.01
25%       0.01
50%       0.01
75%       0.01
max       0.01
Name: outstanding_balance, dtype: float64

**Reading the output.** The number of overpaid invoices (**52**), the **total rupees overpaid**, and a statistical description of the overpayments (smallest, median, largest). Overpayments could be refunds owed, duplicated receipts or data-entry errors. The cause cannot be decided from the data alone, so they are **flagged** for a business answer.

### 3.6 Follow-up: cancelled and delivered orders vs invoices

Invoices should exist for delivered orders only. This cell checks that cancelled orders were never invoiced.

In [28]:
orders_vs_invoices = con.execute("""
    SELECT h.order_status,
           COUNT(*) AS orders,
           SUM(CASE WHEN EXISTS (SELECT 1 FROM invoices i WHERE i.so_id = h.so_id)
                    THEN 1 ELSE 0 END) AS with_invoice
    FROM sales_orders_header h
    GROUP BY h.order_status
""").df()
orders_vs_invoices

,order_status,orders,with_invoice
0,Delivered,18033,18033.0
1,Cancelled,1967,0.0


**Reading the output.** Orders by status and how many have an invoice. **Delivered: 18,033 orders, all with an invoice** (matching the 18,033 invoice rows). **Cancelled: 1,967 orders, none invoiced.** This is the expected business behaviour, so cancelled orders are excluded from revenue and invoice measures through a flag in the processed layer.

### Task 3: Summary

| Check | Result | Decision |
|---|---|---|
| Line arithmetic and GST (sales and purchase) | 0 exceptions; only 18% and 28% rates | **ACCEPT** |
| Header vs lines (sales 20,000; purchase 24,000) | All match | **ACCEPT** |
| Invoices vs payments | 393 ambiguous (duplicate IDs), 52 overpaid, 67 stored-`Paid` under-paid | **FLAG**; derive status from payments (**FIX**) |
| Customer `total_purchase_value` | Almost all customers mismatch; stored ≈ 5% of actual | **FIX**: recomputed field added, original kept and flagged |
| Cancelled orders vs invoices | 1,967 cancelled, none invoiced | **EXCLUDE** from revenue and invoice measures (flag) |

**Correction to an earlier reading.** The first version of this check reported many "Unpaid invoices with receipts". That came from joining on the reused `invoice_id`. With payments summed first and ambiguous invoices set aside, stored `Unpaid` and `Partially Paid` always agree with the payments.

**Status: COMPLETED; EXCEPTIONS FLAGGED.** The money fields are consistent within every order. The finance exceptions come from duplicate IDs, overpayments and a stale customer summary field.

---
## 4. Task 4: Referential integrity and business-rule checks

**Objective.** Check that the tables link to each other correctly (**referential integrity**) and that records obey the business rules.

**Why it matters.** A broken link silently drops rows from every join; a broken rule (a delivery before its order) corrupts every time-based KPI.

**Four parts.**

| Part | Check |
|---|---|
| 1 | Primary keys unique; all 17 foreign keys resolve, with no orphans and no nulls |
| 2 | Cancelled purchase orders have no `received_date`; active ones do |
| 3 | `delivery_date >= order_date` for sales orders |
| 4 | Customer regions with no local branch (informational, since serving a region without a branch is not automatically an error) |

Findings are **identified, investigated and documented**; nothing is corrected here.

### Part 1: Keys

In [29]:
def validate_foreign_keys(con) -> pd.DataFrame:
    rows = []
    for parent, pkey, child, ckey in FK_RELATIONSHIPS:
        total, nulls, orphans = con.execute(f"""
            SELECT COUNT(*),
                   SUM(CASE WHEN c.{ckey} IS NULL THEN 1 ELSE 0 END),
                   SUM(CASE WHEN c.{ckey} IS NOT NULL AND NOT EXISTS (
                            SELECT 1 FROM {parent} p WHERE p.{pkey} = c.{ckey}
                       ) THEN 1 ELSE 0 END)
            FROM {child} c
        """).fetchone()
        rows.append({"Parent_Table": parent, "Child_Table": child,
                     "Parent_Key": pkey, "Child_Key": ckey,
                     "Total_Records": int(total), "Null_Keys": int(nulls or 0),
                     "Orphan_Records": int(orphans or 0)})
    return pd.DataFrame(rows)


def validate_primary_keys(con) -> pd.DataFrame:
    """Duplicate_Records = surplus rows beyond one per key value."""
    rows = []
    for table, keys in PRIMARY_KEYS.items():
        cols = ", ".join(keys)
        null_cond = " OR ".join(f"{k} IS NULL" for k in keys)
        total, nulls, distinct = con.execute(f"""
            SELECT COUNT(*),
                   SUM(CASE WHEN {null_cond} THEN 1 ELSE 0 END),
                   (SELECT COUNT(*) FROM (SELECT DISTINCT {cols} FROM {table}))
            FROM {table}
        """).fetchone()
        rows.append({"Table": table, "Candidate_PK": cols,
                     "Total_Records": int(total), "Null_Records": int(nulls or 0),
                     "Duplicate_Records": int(total - distinct)})
    return pd.DataFrame(rows)


def duplicate_key_details(con, table: str, key: str) -> pd.DataFrame:
    """All rows whose key appears more than once."""
    return con.execute(f"""
        SELECT t.*, d.occurrences AS duplicate_count
        FROM {table} t
        JOIN (SELECT {key}, COUNT(*) AS occurrences FROM {table}
              GROUP BY {key} HAVING COUNT(*) > 1) d
          ON t.{key} = d.{key}
        ORDER BY t.{key}
    """).df()


def classify_duplicate_keys(details: pd.DataFrame, key: str) -> dict:
    """Exact duplicate rows vs same-key-different-content rows, plus frequency."""
    cols = [c for c in details.columns if c != "duplicate_count"]
    exact = int(details.duplicated(subset=cols, keep=False).sum())
    freq = (details.groupby(key).size().value_counts().sort_index()
                   .rename_axis("occurrences").reset_index(name="number_of_ids"))
    return {"duplicate_ids": int(details[key].nunique()),
            "rows_involved": len(details),
            "exact_duplicate_rows": exact,
            "different_content_rows": len(details) - exact,
            "frequency": freq}

**Reading the output.** No output: four functions. `validate_foreign_keys` counts, for each parent→child pair, total rows, null keys and **orphans** (child rows whose parent does not exist). `validate_primary_keys` counts nulls and surplus duplicate rows per candidate key. `duplicate_key_details` pulls every row of a duplicated key, and `classify_duplicate_keys` separates **exact copies** from rows that share a key but differ in content.

In [30]:
fk_summary = validate_foreign_keys(con)
fk_summary

,Parent_Table,Child_Table,Parent_Key,Child_Key,Total_Records,Null_Keys,Orphan_Records
0,branches,inventory_master,branch_id,branch_id,180,0,0
1,products,inventory_master,product_id,product_id,180,0,0
2,suppliers,purchase_orders_header,supplier_id,supplier_id,24000,0,0
3,branches,purchase_orders_header,branch_id,branch_id,24000,0,0
4,purchase_orders_header,purchase_orders_lines,po_id,po_id,155495,0,0
5,products,purchase_orders_lines,product_id,product_id,155495,0,0
6,customers,sales_orders_header,customer_id,customer_id,20000,0,0
7,branches,sales_orders_header,branch_id,branch_id,20000,0,0
8,sales_orders_header,sales_orders_lines,so_id,so_id,130402,0,0
9,products,sales_orders_lines,product_id,product_id,130402,0,0


**Reading the output.** A 17-row table, one per foreign key, e.g. `branches → inventory_master (branch_id)`. **`Null_Keys` and `Orphan_Records` are 0 in every row**, so all 17 relationships are valid: every inventory row has a branch and a product, every sales line has an order and a product, every payment has an invoice, and so on. The candidate keys from the column names are confirmed as real links.

In [31]:
pk_summary = validate_primary_keys(con)
pk_summary

,Table,Candidate_PK,Total_Records,Null_Records,Duplicate_Records
0,branches,branch_id,6,0,0
1,products,product_id,30,0,0
2,inventory_master,"product_id, branch_id",180,0,0
3,suppliers,supplier_id,8,0,0
4,customers,customer_id,500,0,0
5,invoices,invoice_id,18033,0,197
6,payments,payment_id,19257,0,202
7,purchase_orders_header,po_id,24000,0,0
8,purchase_orders_lines,"po_id, line_number",155495,0,0
9,sales_orders_header,so_id,20000,0,0


**Reading the output.** A 12-row table, one per table, with the candidate primary key, row count, null keys and `Duplicate_Records` (surplus rows beyond one per key). **Ten tables show 0.** Two do not: **`invoices` shows 197** and **`payments` shows 202**. For those two tables the supposed primary key is not unique. The next cell shows what the duplicates are.

In [32]:
duplicates = {}
for table, key in (("invoices", "invoice_id"), ("payments", "payment_id")):
    details = duplicate_key_details(con, table, key)
    duplicates[table] = {"details": details, **classify_duplicate_keys(details, key)}
    info = duplicates[table]
    print(f"{table}: {info['duplicate_ids']} duplicate IDs, {info['rows_involved']} rows, "
          f"{info['exact_duplicate_rows']} exact copies, "
          f"{info['different_content_rows']} with different content")
    display(info["frequency"])

invoices: 196 duplicate IDs, 393 rows, 0 exact copies, 393 with different content


,occurrences,number_of_ids
0,2,195
1,3,1


payments: 201 duplicate IDs, 403 rows, 0 exact copies, 403 with different content


,occurrences,number_of_ids
0,2,200
1,3,1


**Reading the output.** For invoices and payments, one summary line and a frequency table each.

| | Duplicate IDs | Rows involved | Exact copies | Different content |
|---|---:|---:|---:|---:|
| `invoice_id` | 196 | 393 | **0** | 393 |
| `payment_id` | 201 | 403 | **0** | 403 |

**There are no exact copies.** Each repeated ID belongs to *different* records (different order, customer, date and amount): the ID was **re-issued**, not the row repeated. That explains why the Week 1 duplicate-row profile found 0 duplicates. Most IDs appear twice; one of each appears three times (the frequency tables). Deleting the extra rows would destroy real transactions, so they are **flagged, not removed**.

### Part 2: Cancelled purchase orders
**Rule:** a cancelled PO has no `received_date`; an active PO should have one.

In [33]:
def cancelled_po_checks(con) -> dict:
    status = con.execute("""
        SELECT po_status, COUNT(*) AS total_pos,
               SUM(CASE WHEN received_date IS NULL THEN 1 ELSE 0 END)     AS null_received,
               SUM(CASE WHEN received_date IS NOT NULL THEN 1 ELSE 0 END) AS has_received
        FROM purchase_orders_header GROUP BY po_status ORDER BY total_pos DESC
    """).df()
    rules = con.execute(f"""
        SELECT
          SUM(CASE WHEN LOWER(TRIM(po_status)) IN {CANCELLED} AND received_date IS NOT NULL THEN 1 ELSE 0 END) AS cancelled_with_received_date,
          SUM(CASE WHEN LOWER(TRIM(po_status)) NOT IN {CANCELLED} AND received_date IS NULL THEN 1 ELSE 0 END) AS active_missing_received_date,
          SUM(CASE WHEN received_date < order_date THEN 1 ELSE 0 END)            AS received_before_order,
          SUM(CASE WHEN expected_delivery_date < order_date THEN 1 ELSE 0 END)   AS expected_before_order
        FROM purchase_orders_header
    """).df()
    exceptions = con.execute(f"""
        SELECT po_id, po_status, order_date, expected_delivery_date, received_date
        FROM purchase_orders_header
        WHERE (LOWER(TRIM(po_status)) IN {CANCELLED} AND received_date IS NOT NULL)
           OR (LOWER(TRIM(po_status)) NOT IN {CANCELLED} AND received_date IS NULL)
           OR received_date < order_date
        LIMIT 20
    """).df()
    return {"status": status, "rules": rules, "exceptions": exceptions}

**Reading the output.** No output: one function returning three tables: PO status with null / non-null received dates, a one-row rule table (cancelled with a date, active without a date, received before ordered, expected before ordered) and a list of any exceptions.

In [34]:
po_checks = cancelled_po_checks(con)
display(po_checks["status"])
display(po_checks["rules"])
po_checks["exceptions"]

,po_status,total_pos,null_received,has_received
0,Received,21630,0.0,21630.0
1,Cancelled,2370,2370.0,0.0


,cancelled_with_received_date,active_missing_received_date,received_before_order,expected_before_order
0,0.0,0.0,0.0,0.0


,po_id,po_status,order_date,expected_delivery_date,received_date


**Reading the output.** Three tables. **(1)** Received: 21,630 POs, 0 null received dates. Cancelled: 2,370 POs, all null. **(2)** The rule table: **all four columns should be 0** (no cancelled PO with a date, no active PO without one, no PO received or expected before it was ordered). **(3)** The exceptions list should be **empty**. This confirms the Task 1 finding with a formal rule.

### Part 3: Delivery dates
**Rule:** `delivery_date >= order_date`, split by order status.

In [35]:
def delivery_date_checks(con) -> dict:
    by_status = con.execute("""
        SELECT order_status, COUNT(*) AS total_orders,
               SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END)           AS missing_order_date,
               SUM(CASE WHEN delivery_date IS NULL THEN 1 ELSE 0 END)        AS missing_delivery_date,
               SUM(CASE WHEN delivery_date <  order_date THEN 1 ELSE 0 END)  AS delivery_before_order,
               SUM(CASE WHEN delivery_date >= order_date THEN 1 ELSE 0 END)  AS valid_dates
        FROM sales_orders_header GROUP BY order_status ORDER BY total_orders DESC
    """).df()
    dates = con.execute(
        "SELECT so_id, order_status, order_date, delivery_date FROM sales_orders_header"
    ).df()
    dates["days_to_deliver"] = (pd.to_datetime(dates["delivery_date"])
                                - pd.to_datetime(dates["order_date"])).dt.days
    is_cancelled = dates["order_status"].str.strip().str.lower().isin(["cancelled", "canceled"])
    return {
        "by_status": by_status,
        "days_summary": dates.groupby("order_status")["days_to_deliver"].describe().round(1),
        "exceptions": dates[dates["days_to_deliver"] < 0],
        "cancelled_with_delivery": dates[is_cancelled & dates["delivery_date"].notna()],
    }

**Reading the output.** No output: one function returning four tables: checks by status, days-to-deliver statistics by status, any deliveries before order, and cancelled orders that still carry a delivery date.

In [40]:
delivery_checks = delivery_date_checks(con)
display(delivery_checks["by_status"])
display(delivery_checks["days_summary"])
print("Delivery before order:", len(delivery_checks["exceptions"]))
print("Cancelled orders with a delivery date:", len(delivery_checks["cancelled_with_delivery"]))

,order_status,total_orders,missing_order_date,missing_delivery_date,delivery_before_order,valid_dates
0,Delivered,18033,0.0,0.0,0.0,18033.0
1,Cancelled,1967,0.0,0.0,0.0,1967.0


,count,mean,std,min,25%,50%,75%,max
order_status,,,,,,,,
Cancelled,1967.0,8.1,4.4,0.0,4.0,8.0,12.0,15.0
Delivered,18033.0,8.1,4.3,0.0,4.0,8.0,12.0,15.0


Delivery before order: 0
Cancelled orders with a delivery date: 1967


**Reading the output.** Two tables and two counts.
* **By status:** for Delivered and Cancelled orders: missing dates, deliveries before order and valid dates. No order lacks a date.
* **Days to deliver:** count, mean, minimum, quartiles and maximum per status. The **minimum should not be negative.**
* **`Delivery before order: 0`**: the business rule holds.
* **`Cancelled orders with a delivery date: 1967`**: every cancelled order still has a delivery date. A cancelled order was never delivered, so that date is a **placeholder**, not a real delivery. It must not be used in order-to-delivery measures.

### Part 4: Customer regions without a local branch
An unmatched region is not automatically an error: a company may serve customers where it has no branch.

In [36]:
def customer_region_checks(con) -> dict:
    no_branch = """NOT EXISTS (SELECT 1 FROM branches b2
                   WHERE LOWER(TRIM(b2.region)) = LOWER(TRIM(c.region)))"""
    return {
        "customer_regions": con.execute(
            "SELECT region, COUNT(*) AS customers FROM customers "
            "GROUP BY region ORDER BY customers DESC").df(),
        "branch_regions": con.execute(
            "SELECT region, COUNT(*) AS branches FROM branches GROUP BY region").df(),
        "unmatched": con.execute(f"""
            SELECT c.region, COUNT(*) AS customers FROM customers c
            WHERE c.region IS NOT NULL AND TRIM(c.region) <> '' AND {no_branch}
            GROUP BY c.region ORDER BY customers DESC""").df(),
        "unmatched_service": con.execute(f"""
            SELECT c.region AS customer_region, c.branch_id,
                   b.region AS serving_branch_region, COUNT(*) AS customers
            FROM customers c JOIN branches b ON c.branch_id = b.branch_id
            WHERE {no_branch}
            GROUP BY c.region, c.branch_id, b.region ORDER BY customers DESC""").df(),
        "region_mismatch": con.execute("""
            SELECT c.region AS customer_region, b.region AS assigned_branch_region,
                   COUNT(*) AS customers
            FROM customers c JOIN branches b ON c.branch_id = b.branch_id
            WHERE LOWER(TRIM(c.region)) <> LOWER(TRIM(b.region))
            GROUP BY c.region, b.region ORDER BY customers DESC""").df(),
    }

**Reading the output.** No output: one function returning five tables: customers per region, branches per region, customer regions with no branch, which branch serves them, and customers assigned to a branch in a different region.

In [37]:
region_checks = customer_region_checks(con)
display(region_checks["customer_regions"])
display(region_checks["branch_regions"])
print("Customer regions with no local branch:")
display(region_checks["unmatched"])
print("Which branch serves them:")
display(region_checks["unmatched_service"])
print("Customers assigned to a branch in another region:")
region_checks["region_mismatch"]

,region,customers
0,South,158
1,West,151
2,East,87
3,North,82
4,Central,11
5,North East,11


,region,branches
0,North,1
1,West,2
2,East,1
3,South,2


Customer regions with no local branch:


,region,customers
0,Central,11
1,North East,11


Which branch serves them:


,customer_region,branch_id,serving_branch_region,customers
0,North East,PUN001,West,5
1,Central,PUN001,West,4
2,North East,KOL001,East,3
3,Central,KOL001,East,3
4,Central,CHN001,South,2
5,North East,AHM001,West,2
6,North East,DEL001,North,1
7,Central,HYD001,South,1
8,Central,AHM001,West,1


Customers assigned to a branch in another region:


,customer_region,assigned_branch_region,customers
0,West,South,13
1,South,East,10
2,North,West,9
3,East,South,9
4,North,South,8
5,North East,West,7
6,North,East,6
7,South,West,6
8,West,East,6
9,East,West,6


**Reading the output.** Five tables, in order.
1. **Customer regions:** how many of the 500 customers are in each region.
2. **Branch regions:** the regions the six branches cover.
3. **Customer regions with no local branch:** per the Week 1 profile these are **Central** and **North East**, **22 customers** in total.
4. **Which branch serves them:** the branches that actually handle those customers, so coverage is not assumed.
5. **Customers assigned to a branch in another region:** the wider cross-region picture.

These customers are real and served by other branches, so this is a **service-coverage note**, not a data error.

### Task 4: Overall findings

In [38]:
def _status(n: int) -> str:
    return "PASSED" if n == 0 else "FLAGGED"


def build_task4_summary(fk: pd.DataFrame, pk: pd.DataFrame, po: dict,
                        delivery: dict, regions: dict) -> pd.DataFrame:
    def dup(table):
        return int(pk.loc[pk["Table"] == table, "Duplicate_Records"].iloc[0])

    fk_issues = int(fk["Orphan_Records"].sum() + fk["Null_Keys"].sum())
    other_pk = int(pk.loc[~pk["Table"].isin(["invoices", "payments"]), "Duplicate_Records"].sum())
    n_unmatched = int(regions["unmatched"]["customers"].sum()) if len(regions["unmatched"]) else 0
    rows = [
        ("1a", "Foreign keys: orphan or null keys (17 relationships)", fk_issues),
        ("1b", "Primary keys: duplicates in the 10 other tables", other_pk),
        ("1c", "Primary key duplicates: invoices.invoice_id", dup("invoices")),
        ("1d", "Primary key duplicates: payments.payment_id", dup("payments")),
        ("2a", "Cancelled POs with a received_date", int(po["rules"]["cancelled_with_received_date"].iloc[0])),
        ("2b", "Active POs missing a received_date", int(po["rules"]["active_missing_received_date"].iloc[0])),
        ("3a", "Deliveries before order date", len(delivery["exceptions"])),
        ("3b", "Cancelled orders carrying a delivery date", len(delivery["cancelled_with_delivery"])),
        ("4", "Customers in regions with no local branch", n_unmatched),
    ]
    out = pd.DataFrame(rows, columns=["Part", "Check", "Exceptions"])
    out["Status"] = out["Exceptions"].map(_status)
    out.loc[out["Part"] == "4", "Status"] = "REVIEW" if n_unmatched else "PASSED"
    return out

**Reading the output.** No output: a helper (`PASSED` if 0 exceptions, otherwise `FLAGGED`) and a function that gathers every Task 4 result into one table. Part 4 is labelled `REVIEW` rather than `FLAGGED`, because an unmatched region is informational.

In [41]:
task4_summary = build_task4_summary(fk_summary, pk_summary, po_checks,
                                    delivery_checks, region_checks)
task4_summary

,Part,Check,Exceptions,Status
0,1a,Foreign keys: orphan or null keys (17 relation...,0,PASSED
1,1b,Primary keys: duplicates in the 10 other tables,0,PASSED
2,1c,Primary key duplicates: invoices.invoice_id,197,FLAGGED
3,1d,Primary key duplicates: payments.payment_id,202,FLAGGED
4,2a,Cancelled POs with a received_date,0,PASSED
5,2b,Active POs missing a received_date,0,PASSED
6,3a,Deliveries before order date,0,PASSED
7,3b,Cancelled orders carrying a delivery date,1967,FLAGGED
8,4,Customers in regions with no local branch,22,REVIEW


**Reading the output.** The Task 4 scorecard, one line per check:

| Check | Exceptions | Status |
|---|---:|---|
| 1a Foreign keys: orphan or null | 0 | PASSED |
| 1b Primary-key duplicates, 10 other tables | 0 | PASSED |
| 1c Duplicates: `invoices.invoice_id` | 197 | FLAGGED |
| 1d Duplicates: `payments.payment_id` | 202 | FLAGGED |
| 2a Cancelled POs with a received date | 0 | PASSED |
| 2b Active POs missing a received date | 0 | PASSED |
| 3a Deliveries before order date | 0 | PASSED |
| 3b Cancelled orders carrying a delivery date | 1,967 | FLAGGED |
| 4 Customers in regions with no local branch | 22 | REVIEW |

### 4.5 Follow-up: Weeks 1–2 status board

This table collects the exceptions from Tasks 1–4 in one view. It is the evidence used to build the issue log in Task 6.

In [42]:
line_checks = pd.concat([line_arithmetic_check(con, "sales"),
                         line_arithmetic_check(con, "purchase")], ignore_index=True)
line_issues = int(line_checks[["line_total_mismatch", "gst_mismatch",
                               "grand_total_mismatch", "unexpected_gst_rate"]].fillna(0).sum().sum())
hv_issues = int((sales_hv["status"] != "Matching").sum()
                + (purchase_hv["status"] != "Matching").sum())

rows = [
    ("Task 1", "Standardisation checks failing", int((task1_validation["status"] != "PASSED").sum())),
    ("Task 2", "Pairs not reachable with signed adjustments", int((~stock_test["plausible_ledger"]).sum())),
    ("Task 2", "Pairs where current_stock differs from ledger", int((~stock_test["ledger_equals_current"]).sum())),
    ("Task 3", "Line arithmetic / GST exceptions (sales + purchase)", line_issues),
    ("Task 3", "Header vs line mismatches (sales + purchase)", hv_issues),
    ("Task 3", "Invoices with duplicate invoice_id (ambiguous)",
        int(invoice_payment["payment_result"].str.startswith("Ambiguous").sum())),
    ("Task 3", "Overpaid invoices", int((invoice_payment["payment_result"] == "Overpaid").sum())),
    ("Task 3", "Stored status contradicts payments", len(contradictions)),
    ("Task 3", "Customers whose stored purchase value does not match", int((customer_recon["status"] == "Mismatching").sum())),
]
status_board = pd.DataFrame(rows, columns=["Task", "Check", "Exceptions"])
status_board["Status"] = status_board["Exceptions"].map(lambda n: "PASSED" if n == 0 else "FLAGGED")

t4 = task4_summary.rename(columns={"Part": "Task"}).assign(Task=lambda d: "Task 4 (" + d["Task"] + ")")
status_board = pd.concat([status_board, t4[["Task", "Check", "Exceptions", "Status"]]], ignore_index=True)
status_board

,Task,Check,Exceptions,Status
0,Task 1,Standardisation checks failing,0,PASSED
1,Task 2,Pairs not reachable with signed adjustments,0,PASSED
2,Task 2,Pairs where current_stock differs from ledger,14,FLAGGED
3,Task 3,Line arithmetic / GST exceptions (sales + purc...,0,PASSED
4,Task 3,Header vs line mismatches (sales + purchase),0,PASSED
5,Task 3,Invoices with duplicate invoice_id (ambiguous),393,FLAGGED
6,Task 3,Overpaid invoices,52,FLAGGED
7,Task 3,Stored status contradicts payments,67,FLAGGED
8,Task 3,Customers whose stored purchase value does not...,500,FLAGGED
9,Task 4 (1a),Foreign keys: orphan or null keys (17 relation...,0,PASSED


**Reading the output.** One table for the whole of Tasks 1–4, with an exception count and a `PASSED` / `FLAGGED` status per check: standardisation checks, stock pairs, line and header checks, the invoice findings, customer values and the Task 4 rows. A `PASSED` row means the rule holds. A `FLAGGED` row is a documented finding that goes into the issue log. Nothing here is hidden or removed.

### Task 4: Summary

| Area | Result | Decision |
|---|---|---|
| Foreign keys | All 17 valid, 0 orphans, 0 nulls | **ACCEPT** |
| Primary keys | Unique in 10 tables; **197** surplus `invoice_id` and **202** surplus `payment_id` rows | **FLAG** |
| Duplicate IDs | No exact copies: re-issued IDs on different records | **FLAG** (do not delete) |
| Cancelled POs | Exactly the POs without a received date | **ACCEPT** |
| Delivery vs order date | No delivery before its order | **ACCEPT** |
| Cancelled orders | All 1,967 still carry a delivery date | **FLAG** |
| Customer regions | 22 customers in regions with no local branch | **ACCEPT** (coverage note) |

**Status: COMPLETED.** Links between tables are sound. The one structural weakness is the reused invoice and payment IDs, which later weeks must handle by joining invoices on `so_id` and payments on `(payment_id, invoice_id)`.

---
## 5. Task 5: Automated data-quality checks and Parquet export

**Objective.** (a) Turn the Task 1–4 checks into **pytest assertions** that run on every data refresh. (b) Write the cleaned tables to **`data/processed/`** as Parquet.

**Why it matters.** A check you run by hand once is a snapshot; a check that runs automatically is a safety net. If next month's refresh brings a duplicate key, an orphan row or a broken GST figure, the suite fails immediately instead of the problem reaching a dashboard. Parquet gives later weeks a fast, typed, compressed copy of the cleaned data, separate from the raw database.

**Deliverables.**

| File | Role |
|---|---|
| `src/cleaning.py` | Reusable pipeline: raw database → cleaned tables → Parquet. Runs with one command: `python -m src.cleaning` |
| `tests/conftest.py` | Shared fixtures: a **read-only** connection to the raw database and the processed folder |
| `tests/test_data_quality.py` | The automated checks |
| `data/processed/*.parquet` | The 12 cleaned tables |

**What the cleaned layer contains.** Every raw column is kept; additions are new columns only.

| Table | Added or changed |
|---|---|
| `branches` | `warehouse_capacity_sqft` |
| `products` | `length_cm`, `width_cm`, `height_cm`, `volume_m3` |
| `customers`, `suppliers` | `pincode` as 6-character text; customers also get `recalc_purchase_value_with_gst`, `recalc_purchase_value_before_gst`, `purchase_value_mismatch_flag` (original kept) |
| `invoices` | `total_paid`, `payment_count`, `outstanding_balance`, `invoice_id_duplicated`, `payment_status_derived` |
| `payments` | `payment_id_duplicated` |
| `sales_orders_header` | `is_cancelled`, `days_to_deliver` |
| `purchase_orders_header` | `is_cancelled` |
| `inventory_master` | `current_above_max_flag` |
| other tables | unchanged (text labels trimmed) |

**Two kinds of test.**
* **Rule tests** assert *zero exceptions*. A failure means a new data problem.
* **Known-issue tests** pin a documented problem to its recorded size (for example, 197 surplus `invoice_id` rows). They pass today but **fail if the problem grows or changes**, which forces a review. They are labelled `KNOWN ISSUE` in the test file and map to the issue log in Task 6.

**No rows are removed.** Decisions such as "exclude cancelled orders from revenue" are carried by flag columns, so the Parquet layer stays complete and reversible.

### 5.1 Create the project folders

In [43]:
from pathlib import Path

for folder in ["src", "tests", "data/processed", "docs"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Folders ready:", [f for f in ["src", "tests", "data/processed", "docs"] if Path(f).is_dir()])

Folders ready: ['src', 'tests', 'data/processed', 'docs']


**Reading the output.** A list of the four folders: `src`, `tests`, `data/processed`, `docs`. Creating them is harmless if they already exist. The next four cells use `%writefile` to save the code **from this notebook into real files**, so the pipeline and tests can run from the command line without the notebook.

### 5.2 The package marker and the cleaning pipeline (`src/cleaning.py`)

In [44]:
%%writefile src/__init__.py
"""Week 2 project package."""

Writing src/__init__.py


**Reading the output.** `Writing src/__init__.py`. This small file lets Python import `src.cleaning`.

In [45]:
%%writefile src/cleaning.py
"""Week 2 cleaning pipeline: raw warehouse.duckdb -> data/processed/*.parquet

Run from the project folder (the one that contains warehouse.duckdb):

    python -m src.cleaning

The raw database is opened read-only and is never modified. Rows are never
deleted: problems are marked with flag columns so every decision stays visible
and reversible (see the Week 2 issue log).
"""
import os
from pathlib import Path

import duckdb
import pandas as pd

DB_PATH = os.environ.get("WAREHOUSE_DB", "warehouse.duckdb")
OUT_DIR = Path(os.environ.get("PROCESSED_DIR", "data/processed"))
MONEY_TOLERANCE = 0.01

ALL_TABLES = [
    "branches", "products", "inventory_master", "suppliers", "customers",
    "invoices", "payments", "purchase_orders_header", "purchase_orders_lines",
    "sales_orders_header", "sales_orders_lines", "stock_ledger",
]

CATEGORICAL_COLUMNS = {
    "branches": ["warehouse_type"],
    "products": ["category", "machine_type"],
    "suppliers": ["supplier_type"],
    "customers": ["customer_type"],
    "invoices": ["payment_status"],
    "purchase_orders_header": ["po_status"],
    "sales_orders_header": ["order_status", "sales_channel"],
    "payments": ["payment_method"],
    "stock_ledger": ["movement_type"],
}

DATE_COLUMNS = {
    "products": ["last_purchase_date"],
    "customers": ["customer_since", "last_purchase_date"],
    "invoices": ["invoice_date", "due_date"],
    "payments": ["payment_date"],
    "purchase_orders_header": ["order_date", "expected_delivery_date", "received_date"],
    "sales_orders_header": ["order_date", "delivery_date"],
    "stock_ledger": ["movement_date"],
}

FK_RELATIONSHIPS = [  # (parent_table, parent_key, child_table, child_key)
    ("branches", "branch_id", "inventory_master", "branch_id"),
    ("products", "product_id", "inventory_master", "product_id"),
    ("suppliers", "supplier_id", "purchase_orders_header", "supplier_id"),
    ("branches", "branch_id", "purchase_orders_header", "branch_id"),
    ("purchase_orders_header", "po_id", "purchase_orders_lines", "po_id"),
    ("products", "product_id", "purchase_orders_lines", "product_id"),
    ("customers", "customer_id", "sales_orders_header", "customer_id"),
    ("branches", "branch_id", "sales_orders_header", "branch_id"),
    ("sales_orders_header", "so_id", "sales_orders_lines", "so_id"),
    ("products", "product_id", "sales_orders_lines", "product_id"),
    ("sales_orders_header", "so_id", "invoices", "so_id"),
    ("customers", "customer_id", "invoices", "customer_id"),
    ("branches", "branch_id", "invoices", "branch_id"),
    ("invoices", "invoice_id", "payments", "invoice_id"),
    ("products", "product_id", "stock_ledger", "product_id"),
    ("branches", "branch_id", "stock_ledger", "branch_id"),
    ("branches", "branch_id", "customers", "branch_id"),
]

PRIMARY_KEYS = {
    "branches": ["branch_id"],
    "products": ["product_id"],
    "inventory_master": ["product_id", "branch_id"],
    "suppliers": ["supplier_id"],
    "customers": ["customer_id"],
    "invoices": ["invoice_id"],
    "payments": ["payment_id"],
    "purchase_orders_header": ["po_id"],
    "purchase_orders_lines": ["po_id", "line_number"],
    "sales_orders_header": ["so_id"],
    "sales_orders_lines": ["so_id", "line_number"],
    "stock_ledger": ["movement_id"],
}

CANCELLED = "('cancelled', 'canceled')"


def connect(db_path: str = DB_PATH) -> duckdb.DuckDBPyConnection:
    """Open the raw warehouse read-only."""
    return duckdb.connect(db_path, read_only=True)


def _pincode_text(series: pd.Series) -> pd.Series:
    """Postal codes are identifiers, not numbers: store as 6-character text."""
    return series.astype("Int64").astype(str).str.zfill(6)


def _trim_categoricals(table: str, df: pd.DataFrame) -> pd.DataFrame:
    for col in CATEGORICAL_COLUMNS.get(table, []):
        df[col] = df[col].str.strip()
    return df


def build_clean_tables(con: duckdb.DuckDBPyConnection) -> dict:
    """Return {table_name: cleaned DataFrame} for all 12 tables.

    Every original column is kept. New columns are additions only.
    """
    tables = {t: con.execute(f"SELECT * FROM {t}").df() for t in ALL_TABLES}

    # ---- Task 1: type casting and standardisation ------------------------
    b = tables["branches"]
    b["warehouse_capacity_sqft"] = (
        b["warehouse_capacity"].str.replace("sqft", "", regex=False).str.strip().astype(int)
    )

    p = tables["products"]
    dims = p["dimensions_cm"].str.extract(
        r"(\d+\.?\d*)\s*x\s*(\d+\.?\d*)\s*x\s*(\d+\.?\d*)"
    )
    dims.columns = ["length_cm", "width_cm", "height_cm"]
    p = pd.concat([p, dims.astype(float)], axis=1)
    p["volume_m3"] = p["length_cm"] * p["width_cm"] * p["height_cm"] / 1_000_000

    tables["products"] = p

    tables["suppliers"]["pincode"] = _pincode_text(tables["suppliers"]["pincode"])

    c = tables["customers"]
    c["pincode"] = _pincode_text(c["pincode"])
    # Task 3: recomputed purchase value (delivered orders only); original kept.
    recalc = con.execute("""
        SELECT customer_id,
               SUM(grand_total)       AS recalc_purchase_value_with_gst,
               SUM(total_order_value) AS recalc_purchase_value_before_gst
        FROM sales_orders_header
        WHERE LOWER(TRIM(order_status)) = 'delivered'
        GROUP BY customer_id
    """).df()
    c = c.merge(recalc, on="customer_id", how="left")
    for col in ["recalc_purchase_value_with_gst", "recalc_purchase_value_before_gst"]:
        c[col] = c[col].fillna(0.0)
    c["purchase_value_mismatch_flag"] = (
        (c["total_purchase_value"] - c["recalc_purchase_value_with_gst"]).abs()
        >= MONEY_TOLERANCE
    )
    tables["customers"] = c

    # ---- Task 3 / 4: flags that carry the issue-log decisions -------------
    tables["invoices"] = con.execute(f"""
        WITH paid AS (
            SELECT invoice_id, SUM(payment_amount) AS total_paid, COUNT(*) AS payment_count
            FROM payments GROUP BY invoice_id
        ), dup AS (
            SELECT invoice_id, COUNT(*) AS invoice_rows FROM invoices GROUP BY invoice_id
        )
        SELECT i.*,
               COALESCE(p.total_paid, 0)    AS total_paid,
               COALESCE(p.payment_count, 0) AS payment_count,
               i.grand_total - COALESCE(p.total_paid, 0) AS outstanding_balance,
               d.invoice_rows > 1 AS invoice_id_duplicated,
               CASE
                 WHEN d.invoice_rows > 1 THEN 'Ambiguous (duplicate invoice_id)'
                 WHEN COALESCE(p.total_paid, 0) <= {MONEY_TOLERANCE} THEN 'Unpaid'
                 WHEN i.grand_total - COALESCE(p.total_paid, 0) >  {MONEY_TOLERANCE} THEN 'Partially paid'
                 WHEN i.grand_total - COALESCE(p.total_paid, 0) < -{MONEY_TOLERANCE} THEN 'Overpaid'
                 ELSE 'Fully paid'
               END AS payment_status_derived
        FROM invoices i
        LEFT JOIN paid p ON i.invoice_id = p.invoice_id
        JOIN dup d       ON i.invoice_id = d.invoice_id
    """).df()

    tables["payments"] = con.execute("""
        SELECT p.*, d.payment_rows > 1 AS payment_id_duplicated
        FROM payments p
        JOIN (SELECT payment_id, COUNT(*) AS payment_rows
              FROM payments GROUP BY payment_id) d
          ON p.payment_id = d.payment_id
    """).df()

    s = tables["sales_orders_header"]
    s["is_cancelled"] = s["order_status"].str.strip().str.lower().isin(["cancelled", "canceled"])
    s["days_to_deliver"] = (
        pd.to_datetime(s["delivery_date"]) - pd.to_datetime(s["order_date"])
    ).dt.days

    po = tables["purchase_orders_header"]
    po["is_cancelled"] = po["po_status"].str.strip().str.lower().isin(["cancelled", "canceled"])

    im = tables["inventory_master"]
    im["current_above_max_flag"] = im["current_stock"] > im["max_stock"]

    # ---- Task 1: categorical clean-up (trim only; no spelling clashes found)
    for name in list(tables):
        tables[name] = _trim_categoricals(name, tables[name])
    return tables


def export_parquet(tables: dict, out_dir=OUT_DIR) -> pd.DataFrame:
    """Write each table to <out_dir>/<table>.parquet and return a summary."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    mem = duckdb.connect()          # separate in-memory connection: raw DB untouched
    rows = []
    for name, df in tables.items():
        path = out_dir / f"{name}.parquet"
        mem.register("tmp_table", df)
        mem.execute(f"COPY tmp_table TO '{path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        mem.unregister("tmp_table")
        rows.append({"table": name, "rows": len(df), "columns": df.shape[1],
                     "file_size_kb": round(path.stat().st_size / 1024, 1)})
    return pd.DataFrame(rows)


def run_pipeline(db_path: str = DB_PATH, out_dir=OUT_DIR) -> pd.DataFrame:
    con = connect(db_path)
    try:
        return export_parquet(build_clean_tables(con), out_dir)
    finally:
        con.close()


if __name__ == "__main__":
    print(run_pipeline().to_string(index=False))

Writing src/cleaning.py


**Reading the output.** `Writing src/cleaning.py`. The file holds, in order: the shared rule lists (tables, categories, dates, foreign keys, primary keys); `build_clean_tables` (all Task 1–4 fixes and flags, returning one DataFrame per table); `export_parquet` (writes each table with ZSTD compression and returns a summary of rows, columns and file size); and `run_pipeline`, which does both from a read-only connection. The Parquet files are written through a **separate in-memory connection**, so the raw database cannot be altered.

### 5.3 The test suite (`tests/`)

In [46]:
%%writefile tests/conftest.py
import os
import sys
from pathlib import Path

import duckdb
import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))          # lets tests import src.cleaning

DB_PATH = os.environ.get("WAREHOUSE_DB", str(ROOT / "warehouse.duckdb"))
PROCESSED = Path(os.environ.get("PROCESSED_DIR", ROOT / "data" / "processed"))


@pytest.fixture(scope="session")
def con():
    """Raw warehouse, opened read-only so a test can never change it."""
    connection = duckdb.connect(DB_PATH, read_only=True)
    yield connection
    connection.close()


@pytest.fixture(scope="session")
def processed_dir():
    return PROCESSED

Writing tests/conftest.py


**Reading the output.** `Writing tests/conftest.py`. The file provides the shared **`con`** fixture, a read-only connection to the raw database (a test physically cannot change data), and the `processed_dir` fixture. Paths can be overridden with the `WAREHOUSE_DB` and `PROCESSED_DIR` environment variables, so the same suite works on any refreshed copy.

In [47]:
%%writefile tests/test_data_quality.py
"""Week 2 automated data-quality tests (pytest).

Run on every data refresh:   python -m pytest tests -v

Two kinds of test:
  * RULE tests assert zero exceptions (a failure is a new data problem).
  * KNOWN-ISSUE tests pin a documented problem to its recorded size, so the
    suite stays green today but fails if the problem grows or changes.
    These are marked KNOWN ISSUE and map to the issue log (Task 6).
"""
import pytest

from src.cleaning import (
    ALL_TABLES, CANCELLED, CATEGORICAL_COLUMNS, DATE_COLUMNS,
    FK_RELATIONSHIPS, PRIMARY_KEYS,
)

TOL = 0.01

EXPECTED_ROWS = {
    "branches": 6, "products": 30, "inventory_master": 180, "suppliers": 8,
    "customers": 500, "invoices": 18033, "payments": 19257,
    "purchase_orders_header": 24000, "purchase_orders_lines": 155495,
    "sales_orders_header": 20000, "sales_orders_lines": 130402,
    "stock_ledger": 237230,
}
ALLOWED_NULLS = {("purchase_orders_header", "received_date")}   # cancelled POs

# Recorded sizes of documented (not yet fixed) issues - see the issue log.
KNOWN_DUPLICATE_KEYS = {"invoices": 197, "payments": 202}       # surplus rows
KNOWN_UNMATCHED_REGION_CUSTOMERS = 22
KNOWN_CURRENT_STOCK_NOT_REACHABLE = 9


def scalar(con, sql):
    return con.execute(sql).fetchone()[0]


# --------------------------------------------------------------- structure
@pytest.mark.parametrize("table", ALL_TABLES)
def test_raw_row_count(con, table):
    assert scalar(con, f"SELECT COUNT(*) FROM {table}") == EXPECTED_ROWS[table]


@pytest.mark.parametrize("table", ALL_TABLES)
def test_no_unexpected_nulls(con, table):
    cols = [r[0] for r in con.execute(f"DESCRIBE {table}").fetchall()]
    bad = {c: scalar(con, f"SELECT COUNT(*) - COUNT({c}) FROM {table}")
           for c in cols if (table, c) not in ALLOWED_NULLS}
    assert {c: n for c, n in bad.items() if n} == {}


@pytest.mark.parametrize("table,col", [(t, c) for t, cs in DATE_COLUMNS.items() for c in cs])
def test_date_columns_are_dates(con, table, col):
    types = {row[0]: row[1] for row in con.execute(f"DESCRIBE {table}").fetchall()}
    assert types[col] == "DATE"


@pytest.mark.parametrize("table,col", [(t, c) for t, cs in CATEGORICAL_COLUMNS.items() for c in cs])
def test_categorical_values_have_one_spelling(con, table, col):
    distinct_raw = scalar(con, f"SELECT COUNT(DISTINCT {col}) FROM {table}")
    distinct_clean = scalar(con, f"SELECT COUNT(DISTINCT LOWER(TRIM({col}))) FROM {table}")
    assert distinct_raw == distinct_clean


# -------------------------------------------------------------------- keys
UNIQUE_TABLES = [t for t in PRIMARY_KEYS if t not in KNOWN_DUPLICATE_KEYS]


@pytest.mark.parametrize("table", UNIQUE_TABLES)
def test_primary_key_unique(con, table):
    cols = ", ".join(PRIMARY_KEYS[table])
    total = scalar(con, f"SELECT COUNT(*) FROM {table}")
    distinct = scalar(con, f"SELECT COUNT(*) FROM (SELECT DISTINCT {cols} FROM {table})")
    assert total == distinct


@pytest.mark.parametrize("table", list(KNOWN_DUPLICATE_KEYS))
def test_known_issue_duplicate_ids_not_growing(con, table):
    """KNOWN ISSUE: invoice_id / payment_id are reused by different records."""
    key = PRIMARY_KEYS[table][0]
    total = scalar(con, f"SELECT COUNT(*) FROM {table}")
    distinct = scalar(con, f"SELECT COUNT(DISTINCT {key}) FROM {table}")
    assert total - distinct == KNOWN_DUPLICATE_KEYS[table]


@pytest.mark.parametrize("parent,pkey,child,ckey", FK_RELATIONSHIPS)
def test_foreign_key_has_no_orphans_or_nulls(con, parent, pkey, child, ckey):
    bad = scalar(con, f"""
        SELECT COUNT(*) FROM {child} c
        WHERE c.{ckey} IS NULL
           OR NOT EXISTS (SELECT 1 FROM {parent} p WHERE p.{pkey} = c.{ckey})
    """)
    assert bad == 0


# ----------------------------------------------------------------- finance
@pytest.mark.parametrize("lines,price", [("sales_orders_lines", "unit_price"),
                                         ("purchase_orders_lines", "unit_cost")])
def test_line_arithmetic(con, lines, price):
    bad = scalar(con, f"""
        SELECT COUNT(*) FROM {lines}
        WHERE ABS(line_total - quantity * {price}) >= {TOL}
           OR ABS(gst_amount - line_total * gst_rate / 100.0) >= {TOL}
           OR ABS(line_grand_total - (line_total + gst_amount)) >= {TOL}
    """)
    assert bad == 0


def test_sales_gst_rates_are_18_or_28(con):
    assert scalar(con, "SELECT COUNT(*) FROM sales_orders_lines WHERE gst_rate NOT IN (18, 28)") == 0


@pytest.mark.parametrize("header,lines,key,h_base,h_gst,h_grand", [
    ("sales_orders_header", "sales_orders_lines", "so_id",
     "total_order_value", "total_gst_amount", "grand_total"),
    ("purchase_orders_header", "purchase_orders_lines", "po_id",
     "total_cost", "total_gst_amount", "grand_total"),
])
def test_header_totals_equal_line_totals(con, header, lines, key, h_base, h_gst, h_grand):
    bad = scalar(con, f"""
        SELECT COUNT(*) FROM {header} h
        LEFT JOIN (SELECT {key}, SUM(line_total) b, SUM(gst_amount) g, SUM(line_grand_total) t
                   FROM {lines} GROUP BY {key}) l ON h.{key} = l.{key}
        WHERE l.t IS NULL
           OR ABS(h.{h_base} - l.b) >= {TOL}
           OR ABS(h.{h_gst} - l.g) >= {TOL}
           OR ABS(h.{h_grand} - l.t) >= {TOL}
    """)
    assert bad == 0


def test_no_invoice_for_a_cancelled_order(con):
    bad = scalar(con, f"""
        SELECT COUNT(*) FROM invoices i
        JOIN sales_orders_header h ON i.so_id = h.so_id
        WHERE LOWER(TRIM(h.order_status)) IN {CANCELLED}
    """)
    assert bad == 0


# ---------------------------------------------------------- business rules
def test_cancelled_pos_have_no_received_date(con):
    assert scalar(con, f"""SELECT COUNT(*) FROM purchase_orders_header
        WHERE LOWER(TRIM(po_status)) IN {CANCELLED} AND received_date IS NOT NULL""") == 0


def test_active_pos_have_a_received_date(con):
    assert scalar(con, f"""SELECT COUNT(*) FROM purchase_orders_header
        WHERE LOWER(TRIM(po_status)) NOT IN {CANCELLED} AND received_date IS NULL""") == 0


def test_po_dates_are_in_order(con):
    assert scalar(con, """SELECT COUNT(*) FROM purchase_orders_header
        WHERE received_date < order_date OR expected_delivery_date < order_date""") == 0


def test_delivery_not_before_order(con):
    assert scalar(con, "SELECT COUNT(*) FROM sales_orders_header WHERE delivery_date < order_date") == 0


def test_known_issue_customers_in_regions_without_branch(con):
    """KNOWN ISSUE (accepted): customers in regions with no local branch."""
    n = scalar(con, """
        SELECT COUNT(*) FROM customers c
        WHERE c.region IS NOT NULL AND TRIM(c.region) <> ''
          AND NOT EXISTS (SELECT 1 FROM branches b
                          WHERE LOWER(TRIM(b.region)) = LOWER(TRIM(c.region)))""")
    assert n <= KNOWN_UNMATCHED_REGION_CUSTOMERS


# ------------------------------------------------------------------- stock
def test_stock_movement_types_and_quantities(con):
    types = {r[0] for r in con.execute("SELECT DISTINCT movement_type FROM stock_ledger").fetchall()}
    assert types == {"IN", "OUT", "ADJUSTMENT"}
    assert scalar(con, "SELECT COUNT(*) FROM stock_ledger WHERE quantity <= 0") == 0
    assert scalar(con, "SELECT COUNT(*) FROM inventory_master WHERE opening_stock < 0 OR current_stock < 0") == 0


_EXPECTED_CTE = """
WITH exp AS (
    SELECT i.product_id, i.branch_id, i.current_stock,
           CAST(i.opening_stock
                + COALESCE(SUM(CASE WHEN s.movement_type = 'IN'  THEN s.quantity END), 0)
                - COALESCE(SUM(CASE WHEN s.movement_type = 'OUT' THEN s.quantity END), 0) AS BIGINT) AS expected,
           CAST(COALESCE(SUM(CASE WHEN s.movement_type = 'ADJUSTMENT' THEN s.quantity END), 0) AS BIGINT) AS adj
    FROM inventory_master i
    LEFT JOIN stock_ledger s ON i.product_id = s.product_id AND i.branch_id = s.branch_id
    GROUP BY i.product_id, i.branch_id, i.opening_stock, i.current_stock
)
"""

# A balance B is reachable when |B - expected| <= total adjustments and has the
# same parity (a signed sum of positive quantities).
_LEDGER_NOT_REACHABLE = _EXPECTED_CTE.rstrip() + """, last_day AS (
    SELECT s.product_id, s.branch_id, s.running_balance
    FROM stock_ledger s
    JOIN (SELECT product_id, branch_id, MAX(movement_date) AS d
          FROM stock_ledger GROUP BY product_id, branch_id) l
      ON s.product_id = l.product_id AND s.branch_id = l.branch_id AND s.movement_date = l.d
)
SELECT COUNT(*) FROM exp e
WHERE NOT EXISTS (
    SELECT 1 FROM last_day l
    WHERE l.product_id = e.product_id AND l.branch_id = e.branch_id
      AND ABS(l.running_balance - e.expected) <= e.adj
      AND (e.adj - ABS(l.running_balance - e.expected)) % 2 = 0)
"""

_CURRENT_NOT_REACHABLE = _EXPECTED_CTE + """
SELECT COUNT(*) FROM exp e
WHERE ABS(e.current_stock - e.expected) > e.adj
   OR (e.adj - ABS(e.current_stock - e.expected)) % 2 <> 0
"""


def test_ledger_balance_reachable_with_signed_adjustments(con):
    """Opening + IN - OUT +/- adjustments reaches the ledger balance (order-independent)."""
    assert scalar(con, _LEDGER_NOT_REACHABLE) == 0


def test_known_issue_current_stock_not_reachable(con):
    """KNOWN ISSUE: current_stock disagrees with the ledger for a few pairs."""
    assert scalar(con, _CURRENT_NOT_REACHABLE) <= KNOWN_CURRENT_STOCK_NOT_REACHABLE


# ------------------------------------------------------- processed (parquet)
def _parquet(processed_dir, table):
    path = processed_dir / f"{table}.parquet"
    assert path.exists(), f"{path} is missing - run `python -m src.cleaning` first"
    return path.as_posix()


@pytest.mark.parametrize("table", ALL_TABLES)
def test_processed_row_count_matches_raw(con, processed_dir, table):
    path = _parquet(processed_dir, table)
    assert scalar(con, f"SELECT COUNT(*) FROM read_parquet('{path}')") == EXPECTED_ROWS[table]


def test_processed_capacity_and_volume_are_numeric(con, processed_dir):
    b, p = _parquet(processed_dir, "branches"), _parquet(processed_dir, "products")
    assert scalar(con, f"SELECT COUNT(*) FROM read_parquet('{b}') "
                       "WHERE warehouse_capacity_sqft IS NULL OR warehouse_capacity_sqft <= 0") == 0
    assert scalar(con, f"SELECT COUNT(*) FROM read_parquet('{p}') "
                       "WHERE volume_m3 IS NULL OR volume_m3 <= 0") == 0


@pytest.mark.parametrize("table", ["customers", "suppliers"])
def test_processed_pincode_is_six_digit_text(con, processed_dir, table):
    path = _parquet(processed_dir, table)
    assert scalar(con, f"SELECT COUNT(*) FROM read_parquet('{path}') "
                       "WHERE NOT regexp_matches(CAST(pincode AS VARCHAR), '^[0-9]{6}$')") == 0


def test_processed_invoice_status_is_complete(con, processed_dir):
    path = _parquet(processed_dir, "invoices")
    assert scalar(con, f"SELECT COUNT(*) FROM read_parquet('{path}') "
                       "WHERE payment_status_derived IS NULL") == 0
    ambiguous = scalar(con, f"SELECT COUNT(*) FROM read_parquet('{path}') "
                            "WHERE payment_status_derived LIKE 'Ambiguous%'")
    flagged = scalar(con, f"SELECT COUNT(*) FROM read_parquet('{path}') WHERE invoice_id_duplicated")
    assert ambiguous == flagged


def test_processed_cancelled_flags_match_status(con, processed_dir):
    so = _parquet(processed_dir, "sales_orders_header")
    po = _parquet(processed_dir, "purchase_orders_header")
    assert scalar(con, f"SELECT COUNT(*) FROM read_parquet('{so}') WHERE is_cancelled") == \
        scalar(con, f"SELECT COUNT(*) FROM sales_orders_header WHERE LOWER(TRIM(order_status)) IN {CANCELLED}")
    assert scalar(con, f"SELECT COUNT(*) FROM read_parquet('{po}') WHERE is_cancelled") == \
        scalar(con, f"SELECT COUNT(*) FROM purchase_orders_header WHERE LOWER(TRIM(po_status)) IN {CANCELLED}")

Writing tests/test_data_quality.py


**Reading the output.** `Writing tests/test_data_quality.py`. The suite has about **107 test cases** in these groups:

| Group | Test cases | What is asserted |
|---|---:|---|
| Raw row counts | 12 | Each table has the expected number of rows |
| Unexpected nulls | 12 | Only `received_date` may be null |
| Date columns | 12 | Reviewed date columns are `DATE` type |
| Categories | 11 | One spelling per label |
| Primary keys | 10 + 2 | Unique in 10 tables; `invoice_id` and `payment_id` duplicates pinned at 197 and 202 (**known issue**) |
| Foreign keys | 17 | No orphans or nulls |
| Finance | 6 | Line arithmetic, GST rates, header = lines, no invoice for a cancelled order |
| Business rules | 5 | Cancelled POs, PO dates, delivery after order, region coverage (**known issue**, at most 22) |
| Stock | 3 | Movement types and signs; ledger consistent with signed adjustments; `current_stock` not reachable for at most 9 pairs (**known issue**) |
| Processed Parquet | 17 | Row counts match raw; capacity and volume numeric; postal codes are 6 digits; derived invoice status complete; cancelled flags match status |

### 5.4 Run the cleaning pipeline and write the Parquet files

In [48]:
import importlib, os, sys

sys.path.insert(0, os.getcwd())          # make `src` importable from the notebook
import src.cleaning as cleaning
importlib.reload(cleaning)               # pick up edits if the file was rewritten

export_summary = cleaning.run_pipeline()  # opens its own read-only connection
export_summary

,table,rows,columns,file_size_kb
0,branches,6,14,2.8
1,products,30,27,7.9
2,inventory_master,180,9,3.7
3,suppliers,8,12,2.6
4,customers,500,18,22.7
5,invoices,18033,15,604.6
6,payments,19257,6,277.3
7,purchase_orders_header,24000,11,690.5
8,purchase_orders_lines,155495,9,3788.3
9,sales_orders_header,20000,13,464.5


**Reading the output.** A 12-row table with one line per Parquet file: table name, **rows**, **columns** and **file size in KB**. The row counts should match the raw tables exactly (for example `stock_ledger` 237,230, `purchase_orders_lines` 155,495, `branches` 6), because **no row is ever dropped**. The column counts are higher than the raw tables where flags were added (for example `invoices`: 10 raw columns plus 5 new ones). The largest files are the ledger and the purchase lines.

### 5.5 Confirm what was added to each table

In [49]:
rows = []
for t in cleaning.ALL_TABLES:
    raw_cols = [r[0] for r in con.execute(f"DESCRIBE {t}").fetchall()]
    new_cols = [r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('data/processed/{t}.parquet')").fetchall()]
    assert set(raw_cols) <= set(new_cols), f"{t}: a raw column was lost"
    rows.append({"table": t, "raw_columns": len(raw_cols), "processed_columns": len(new_cols),
                 "columns_added": ", ".join(c for c in new_cols if c not in raw_cols) or "-"})

pd.DataFrame(rows)

,table,raw_columns,processed_columns,columns_added
0,branches,13,14,warehouse_capacity_sqft
1,products,23,27,"length_cm, width_cm, height_cm, volume_m3"
2,inventory_master,8,9,current_above_max_flag
3,suppliers,12,12,-
4,customers,15,18,"recalc_purchase_value_with_gst, recalc_purchas..."
5,invoices,10,15,"total_paid, payment_count, outstanding_balance..."
6,payments,5,6,payment_id_duplicated
7,purchase_orders_header,10,11,is_cancelled
8,purchase_orders_lines,9,9,-
9,sales_orders_header,11,13,"is_cancelled, days_to_deliver"


**Reading the output.** For each table: raw column count, processed column count and the **names of the added columns**. The cell also `assert`s that **no original column was lost**; if it passes silently, the layer is purely additive. Compare the "columns added" list with the table in this section's introduction. Tables with `-` were left unchanged. (`pincode` is converted in place, so it appears as a *changed* column, not an added one.)

### 5.6 Cross-check the Parquet layer against Task 3

In [50]:
processed_status = (con.execute("""
    SELECT payment_status_derived AS payment_result, COUNT(*) AS invoices
    FROM read_parquet('data/processed/invoices.parquet')
    GROUP BY 1 ORDER BY 2 DESC
""").df())
display(processed_status)

from_task3 = invoice_payment["payment_result"].value_counts().to_dict()
from_parquet = dict(zip(processed_status["payment_result"], processed_status["invoices"]))
assert from_task3 == from_parquet, "Parquet invoice status differs from the Task 3 reconciliation"
print("Parquet derived status matches the Task 3 reconciliation exactly.")

,payment_result,invoices
0,Fully paid,12145
1,Partially paid,3645
2,Unpaid,1798
3,Ambiguous (duplicate invoice_id),393
4,Overpaid,52


Parquet derived status matches the Task 3 reconciliation exactly.


**Reading the output.** The derived invoice status counted straight from the Parquet file, which should reproduce the Task 3 table (Fully paid 12,145; Partially paid 3,645; Unpaid 1,798; Ambiguous 393; Overpaid 52). The `assert` then compares the two dictionaries; the closing message appears only if they are **identical**. This proves the pipeline writes exactly what the notebook analysis found, so there is a single version of the truth.

### 5.7 Run the automated tests

The suite is run as a separate process (`python -m pytest`), exactly as it would run after any data refresh.

In [51]:
import subprocess

# pip install pytest   (run once if pytest is not installed)
result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--tb=short",
                         "-p", "no:cacheprovider"],
                        capture_output=True, text=True)
print(result.stdout)
if result.stderr.strip():
    print("STDERR:\n", result.stderr)
print("pytest exit code:", result.returncode, "(0 = every test passed)")

........................................................................ [ 67%]
...................................                                      [100%]
============================== warnings summary ===============================
..\..\..\AppData\Local\Programs\Python\Python313\Lib\site-packages\anyio\pytest_plugin.py:14
  C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\anyio\pytest_plugin.py:14: PytestRemovedIn10Warning: _pytest.python.CallSpec2 has been renamed to CallSpec.
  The CallSpec2 alias will be removed in pytest 10.
  Update imports to use CallSpec instead.
  See https://docs.pytest.org/en/stable/deprecations.html#callspec2-renamed
    from _pytest.python import CallSpec2

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
107 passed, 1 warning in 7.41s

pytest exit code: 0 (0 = every test passed)


**Reading the output.** pytest prints one character per test (`.` pass, `F` fail, `x`/`X` expected failure), then a final line such as **`107 passed in 20s`**, and finally the exit code.

* **Exit code 0 and "N passed":** every rule holds and every known issue is still its documented size.
* **One or more `F`:** a failure block follows with the test name and the numbers involved. A failed **rule** test is a *new* problem. A failed **known-issue** test means a documented problem *changed size*; check whether the data was refreshed, then update the issue log and the pinned number together.

A failure here should be treated as a finding, not something to silence.

### 5.8 Running it on every refresh

After any refresh of `warehouse.duckdb`, two commands rebuild and verify everything from raw data:

```bash
python -m src.cleaning        # raw database -> data/processed/*.parquet
python -m pytest tests -q     # verify raw rules, known issues and the Parquet layer
```

For automation (a scheduled job or a GitHub Actions step), chain them so the second only runs if the first succeeds: `python -m src.cleaning && python -m pytest tests -q`.

### Task 5: Summary

| Deliverable | Result |
|---|---|
| Reusable pipeline | `src/cleaning.py`, run with `python -m src.cleaning`; raw data opened read-only |
| Cleaned layer | 12 Parquet files in `data/processed/`; same row counts as raw; additive columns only |
| Automated checks | about 107 pytest cases over structure, keys, finance, business rules, stock and the Parquet layer |
| Known issues | Pinned to their recorded size so they cannot silently grow |
| Consistency | Parquet invoice status cross-checked against the Task 3 reconciliation |

**Decision principle.** Fix what is certain (types, units, derived fields). Flag what needs a business answer. Never delete a row to make a check pass.

**Status: COMPLETED**, once the pytest cell above reports all tests passed.

---
## 6. Task 6: Issue log and resolution decisions

**Objective.** List every data-quality issue found in Tasks 1–5 and record **one decision** for each.

**Why it matters.** Without a recorded decision, a "known issue" is just a rumour. The log tells every later week which fields to trust, which to avoid and which questions are still open.

**The four decisions.**

| Decision | Meaning | Example in this project |
|---|---|---|
| **FIX** | Correct the problem in the cleaned layer | Capacity text → numeric sqft |
| **FLAG** | Keep the data, mark it, escalate for a business answer | Reused `invoice_id` values |
| **ACCEPT** | A valid or explainable condition; no change | Missing `received_date` on cancelled POs |
| **EXCLUDE** | Keep the row but leave it out of specific analyses (via a flag column) | Cancelled orders in revenue |

**How the log is built.** The *Evidence* column is generated from the live results of Tasks 1–5 (the variables created above), so numbers come from the data, not from memory. Decisions for checks that have no exceptions are set automatically (`ACCEPT` if 0 exceptions, otherwise `FLAG`), so the log cannot drift from the checks.

**Severity** follows the Week 1 scale: **High** (changes a headline conclusion), **Medium** (affects a KPI), **Low** (cosmetic or no analytical impact). **Owner** uses the plan's roles: DA (Data Analyst) or DS (Data Scientist).

### 6.1 Build the issue log

In [52]:
# ---- evidence pulled from the results of Tasks 1-5 -------------------------
def n(x):
    return f"{int(x):,}"

def decide(count, ok="ACCEPT", bad="FLAG"):
    return ok if int(count) == 0 else bad

ms = stock_movement_summary(con).set_index("movement_type")
adj_n = ms.loc["ADJUSTMENT", "total_records"]
in_qty, out_qty = float(ms.loc["IN", "total_quantity"]), float(ms.loc["OUT", "total_quantity"])

vc = invoice_payment["payment_result"].value_counts()
n_amb, n_over = int(vc.get("Ambiguous (duplicate invoice_id)", 0)), int(vc.get("Overpaid", 0))
over_rupees = float((-invoice_payment.loc[invoice_payment["payment_result"] == "Overpaid",
                                          "outstanding_balance"]).sum())
cust_summary = summarise_customer_reconciliation(customer_recon)
canc = orders_vs_invoices[orders_vs_invoices["order_status"].str.strip().str.lower()
                          .isin(["cancelled", "canceled"])]

dup = lambda t: int(pk_summary.loc[pk_summary["Table"] == t, "Duplicate_Records"].iloc[0])
fk_bad = int(fk_summary[["Null_Keys", "Orphan_Records"]].sum().sum())
po_rules = po_checks["rules"].iloc[0][["cancelled_with_received_date", "active_missing_received_date",
                                       "received_before_order", "expected_before_order"]].fillna(0).sum()
n_del_before = len(delivery_checks["exceptions"])
n_canc_deliv = len(delivery_checks["cancelled_with_delivery"])
n_unmatched = int(region_checks["unmatched"]["customers"].sum()) if len(region_checks["unmatched"]) else 0
n_other_region = int(region_checks["region_mismatch"]["customers"].sum()) if len(region_checks["region_mismatch"]) else 0
above_max = int(con.execute("SELECT COUNT(*) FROM inventory_master WHERE current_stock > max_stock").fetchone()[0])
n_inv_pairs = int(con.execute("SELECT COUNT(*) FROM inventory_master").fetchone()[0])
n_curr_diff = len(mismatch)
n_genuine = int((~mismatch["explained_by_ordering"]).sum())
n_curr_unreach = int((~stock_test["plausible_current"]).sum())
n_ledger_bad = int((~stock_test["plausible_ledger"]).sum())
n_last_day = int((stock_test["last_day_rows"] > 1).sum())
n_cat_trim = int(cat_check["needs_cleaning"].sum())
n_cat_clash = len(categorical_inconsistencies(cat_check))
n_other_missing = int(date_check.loc[date_check["column_name"] != "received_date", "missing_dates"].sum())

# ---- the log: (id, task, table/column, severity, issue, evidence, decision, treatment, owner, status)
issues = [
 ("I01", "Task 1", "branches.warehouse_capacity", "Low",
  "Capacity stored as text with a unit ('45230 sqft')",
  f"{len(branches_clean)} branches converted", "FIX",
  "Numeric column warehouse_capacity_sqft added; original text kept", "DA", "Applied in processed layer"),
 ("I02", "Task 1", "products.dimensions_cm", "Low",
  "Three measurements packed in one text field",
  f"{len(products_clean)} products parsed; volume_m3 added", "FIX",
  "length_cm, width_cm, height_cm and volume_m3 added; original kept", "DA", "Applied in processed layer"),
 ("I03", "Task 1", "customers.pincode, suppliers.pincode", "Low",
  "Postal code (an identifier) stored as a number",
  "Both pincode columns converted", "FIX",
  "Stored as 6-character text", "DA", "Applied in processed layer"),
 ("I04", "Task 1", "7 tables, 12 date columns", "Low",
  "Date type and format review",
  f"All {len(date_check)} reviewed columns are DATE; {n(n_other_missing)} missing outside received_date", "ACCEPT",
  "No conversion needed", "DA", "Documented"),
 ("I05", "Task 1", "purchase_orders_header.received_date", "Low",
  "Blank received dates",
  f"{n(date_check.loc[date_check['column_name']=='received_date','missing_dates'].iloc[0])} blanks, all on cancelled POs",
  "ACCEPT", "Valid business blank; no date invented", "DA", "Documented"),
 ("I06", "Task 1/4", "purchase_orders_header (cancelled POs)", "Low",
  "Cancelled POs are never received and would distort lead-time measures",
  f"{n(po_rules.sum())} rule exceptions; cancelled flag added", "EXCLUDE",
  "is_cancelled flag: leave cancelled POs out of lead-time, delay and on-time measures", "DA", "Flag column in processed layer"),
 ("I07", "Task 1", "9 tables, 11 category columns", "Low",
  "Spacing or spelling variants in labels",
  f"{n_cat_trim} values needing trim; {n_cat_clash} spelling clashes", decide(n_cat_trim + n_cat_clash),
  "Labels trimmed in the pipeline as a safeguard", "DA", "Documented"),
 ("I08", "Task 2", "stock_ledger.quantity (ADJUSTMENT)", "Medium",
  "Adjustment quantities are all positive, so gain vs loss is not stored",
  f"{n(adj_n)} adjustment records; direction known only in aggregate", "FLAG",
  "Adjustments kept separate from IN/OUT; direction must be confirmed before shrinkage analysis (Week 11)", "DS", "Open: needs business input"),
 ("I09", "Task 2", "stock_ledger.running_balance", "Medium",
  "Row order within a day is not stored; movement_id sorts as text",
  f"{n_last_day} product-branch pairs have several movements on their last date", "ACCEPT",
  "Order-independent test used. The earlier 179/180 mismatches were an ordering artefact", "DS", "Documented"),
 ("I10", "Task 2", "stock_ledger (final balance)", "Medium",
  "Is the ledger internally consistent?",
  f"{n_inv_pairs - n_ledger_bad} of {n_inv_pairs} pairs consistent with signed adjustments", decide(n_ledger_bad),
  "Ledger used as the stock source of truth", "DS", "Documented"),
 ("I11", "Task 2", "inventory_master.current_stock", "Medium",
  "current_stock differs from the ledger balance",
  f"{n_curr_diff} pairs differ; {n_genuine} not explained by last-day ordering; {n_curr_unreach} not reachable", decide(n_curr_diff),
  "Rebuild stock from the ledger; keep current_stock for comparison", "DS", "Open: investigate genuine differences"),
 ("I12", "Task 3", "sales and purchase lines", "Medium",
  "Line total, GST and grand-total arithmetic",
  f"{n(line_issues)} exceptions across sales and purchase lines", decide(line_issues),
  "None needed", "DA", "Documented"),
 ("I13", "Task 3", "sales and purchase headers", "Medium",
  "Header totals vs the sum of their lines",
  f"{n(hv_issues)} mismatching headers", decide(hv_issues),
  "None needed", "DA", "Documented"),
 ("I14", "Task 3/4", "invoices.invoice_id", "High",
  "invoice_id re-issued on different invoices (no exact copies)",
  f"{n(dup('invoices'))} surplus rows; {n(n_amb)} invoices ambiguous", "FLAG",
  "Rows kept. invoice_id_duplicated flag added; join invoices on so_id, not invoice_id", "DA", "Flag column in processed layer"),
 ("I15", "Task 4", "payments.payment_id", "Medium",
  "payment_id re-issued on different payments (no exact copies)",
  f"{n(dup('payments'))} surplus rows", "FLAG",
  "Rows kept. payment_id_duplicated flag added; use (payment_id, invoice_id) as the key", "DA", "Flag column in processed layer"),
 ("I16", "Task 3", "invoices.payment_status", "High",
  "Stored status contradicts the payments",
  f"{n(len(contradictions))} stored 'Paid' invoices are only partly paid", "FIX",
  "payment_status_derived computed from payments; stored status kept for comparison", "DA", "Applied in processed layer"),
 ("I17", "Task 3", "invoices vs payments", "High",
  "Invoices paid more than their grand total",
  f"{n(n_over)} overpaid invoices; INR {over_rupees:,.0f} in total", "FLAG",
  "Reported as 'Overpaid'; cause (refund, duplicate receipt or error) needs a business answer", "DA", "Open: needs business input"),
 ("I18", "Task 3", "customers.total_purchase_value", "High",
  "Stored value does not match recomputed sales",
  f"{n(cust_summary['mismatching'])} of {n(cust_summary['customers'])} customers; stored = {cust_summary['stored_as_pct_of_recomputed']}% of recomputed", "FIX",
  "Recomputed value from delivered orders added; mismatch flag added; original kept but not used", "DA", "Applied in processed layer"),
 ("I19", "Task 3", "sales_orders_header (cancelled)", "Low",
  "Cancelled orders are not invoiced and are not revenue",
  f"{n(canc['orders'].sum())} cancelled orders; {n(canc['with_invoice'].sum())} invoiced", "EXCLUDE",
  "is_cancelled flag: exclude from revenue, invoice and delivery measures", "DA", "Flag column in processed layer"),
 ("I20", "Task 4", "sales_orders_header.delivery_date", "Medium",
  "Cancelled orders still carry a delivery date",
  f"{n(n_canc_deliv)} cancelled orders", decide(n_canc_deliv),
  "Treat the date as a placeholder; exclude cancelled orders from order-to-delivery measures", "DA", "Flag column in processed layer"),
 ("I21", "Task 4", "17 foreign keys", "High",
  "Orphan or null foreign keys",
  f"{n(fk_bad)} orphan or null keys across 17 relationships", decide(fk_bad),
  "None needed; joins are safe", "DA", "Documented"),
 ("I22", "Task 4", "sales_orders_header dates", "Medium",
  "Delivery before order date",
  f"{n(n_del_before)} orders", decide(n_del_before),
  "None needed", "DA", "Documented"),
 ("I23", "Task 4", "customers.region vs branches.region", "Low",
  "Customers in regions without a local branch",
  f"{n(n_unmatched)} customers; {n(n_other_region)} assigned to a branch in another region", "ACCEPT",
  "Served by other branches: a service-coverage note, not a data error", "DA", "Documented"),
 ("I24", "Week 1/5", "inventory_master.current_stock vs max_stock", "High",
  "Current stock above the maximum stock level",
  f"{n(above_max)} of {n(n_inv_pairs)} product-branch pairs", "FLAG",
  "current_above_max_flag added; headline overstock finding for Week 5", "DS", "Flag column in processed layer"),
 ("I25", "Task 2", "stock_ledger IN vs OUT", "High",
  "Stock inflow far exceeds outflow",
  f"IN {in_qty:,.0f} vs OUT {out_qty:,.0f} units (about {in_qty/out_qty:.0f}:1)", "FLAG",
  "Root-cause analysis in Week 5; drives the inventory optimisation recommendations", "DS", "Open: analytical finding"),
 ("I26", "Week 1", "products vs inventory_master", "Medium",
  "Two sets of reorder, safety and max parameters (company level and branch level)",
  "Per the Week 1 profile", "ACCEPT",
  "Use branch-level parameters for stock health; compare both with optimised values", "DS", "Documented"),
 ("I27", "Week 1", "suppliers.china_tax_id", "Low",
  "Tax-ID city prefix does not match the supplier city",
  "Per the Week 1 profile", "ACCEPT",
  "Master-data note; no analytical impact", "DA", "Documented"),
 ("I28", "Task 3", "sales_orders_lines.unit_price, line_total", "Low",
  "Whole-number types for money fields",
  "Line arithmetic reconciles exactly, so no precision is lost", "ACCEPT",
  "Keep as stored", "DA", "Documented"),
]

issue_log = pd.DataFrame(issues, columns=["Issue_ID", "Task", "Table_or_Column", "Severity", "Issue",
                                          "Evidence", "Decision", "Treatment_or_Rationale", "Owner", "Status"])
pd.set_option("display.max_colwidth", 90)
issue_log

,Issue_ID,Task,Table_or_Column,Severity,Issue,Evidence,Decision,Treatment_or_Rationale,Owner,Status
0,I01,Task 1,branches.warehouse_capacity,Low,Capacity stored as text with a unit ('45230 sqft'),6 branches converted,FIX,Numeric column warehouse_capacity_sqft added; original text kept,DA,Applied in processed layer
1,I02,Task 1,products.dimensions_cm,Low,Three measurements packed in one text field,30 products parsed; volume_m3 added,FIX,"length_cm, width_cm, height_cm and volume_m3 added; original kept",DA,Applied in processed layer
2,I03,Task 1,"customers.pincode, suppliers.pincode",Low,Postal code (an identifier) stored as a number,Both pincode columns converted,FIX,Stored as 6-character text,DA,Applied in processed layer
3,I04,Task 1,"7 tables, 12 date columns",Low,Date type and format review,All 12 reviewed columns are DATE; 0 missing outside received_date,ACCEPT,No conversion needed,DA,Documented
4,I05,Task 1,purchase_orders_header.received_date,Low,Blank received dates,"2,370 blanks, all on cancelled POs",ACCEPT,Valid business blank; no date invented,DA,Documented
5,I06,Task 1/4,purchase_orders_header (cancelled POs),Low,Cancelled POs are never received and would distort lead-time measures,0 rule exceptions; cancelled flag added,EXCLUDE,"is_cancelled flag: leave cancelled POs out of lead-time, delay and on-time measures",DA,Flag column in processed layer
6,I07,Task 1,"9 tables, 11 category columns",Low,Spacing or spelling variants in labels,0 values needing trim; 0 spelling clashes,ACCEPT,Labels trimmed in the pipeline as a safeguard,DA,Documented
7,I08,Task 2,stock_ledger.quantity (ADJUSTMENT),Medium,"Adjustment quantities are all positive, so gain vs loss is not stored","2,454 adjustment records; direction known only in aggregate",FLAG,Adjustments kept separate from IN/OUT; direction must be confirmed before shrinkage an...,DS,Open: needs business input
8,I09,Task 2,stock_ledger.running_balance,Medium,Row order within a day is not stored; movement_id sorts as text,33 product-branch pairs have several movements on their last date,ACCEPT,Order-independent test used. The earlier 179/180 mismatches were an ordering artefact,DS,Documented
9,I10,Task 2,stock_ledger (final balance),Medium,Is the ledger internally consistent?,180 of 180 pairs consistent with signed adjustments,ACCEPT,Ledger used as the stock source of truth,DS,Documented


**Reading the output.** The full issue log: **28 rows**, one per issue. Read each row left to right: *where* the issue is, its *severity*, a plain description, the *evidence* (a number taken live from the earlier results), the *decision* and what was done about it. The *Status* column separates fixes already built into the Parquet layer (**Applied**), decisions carried by a flag column (**Flag column**), items that still need a business answer (**Open**) and conditions simply recorded (**Documented**). Rows I12, I13, I21 and I22 are generated automatically: they show `ACCEPT` only if the check really found 0 exceptions.

### 6.2 Validate the log and summarise the decisions

In [53]:
allowed = {"FIX", "FLAG", "ACCEPT", "EXCLUDE"}
assert issue_log["Issue_ID"].is_unique, "duplicate issue ids"
assert set(issue_log["Decision"]) <= allowed, "a decision is not one of FIX / FLAG / ACCEPT / EXCLUDE"
assert issue_log[["Issue", "Evidence", "Treatment_or_Rationale", "Owner"]].notna().all().all(), "blank field in the log"
print("Issues logged:", len(issue_log), "| every issue has a valid decision.\n")

display(issue_log["Decision"].value_counts().rename("issues").to_frame())
pd.crosstab(issue_log["Severity"], issue_log["Decision"], margins=True)

Issues logged: 28 | every issue has a valid decision.



,issues
Decision,
ACCEPT,13
FLAG,8
FIX,5
EXCLUDE,2


Decision,ACCEPT,EXCLUDE,FIX,FLAG,All
Severity,,,,,
High,1,0,2,4,7
Low,6,2,3,0,11
Medium,6,0,0,4,10
All,13,2,5,8,28


**Reading the output.** The three `assert` lines are the log's own quality gate: each issue has a unique ID, a decision from the allowed four, and no blank fields. Execution stops with a message if any fails. Then two tables: the **number of issues per decision** (how much was fixed, flagged, accepted or excluded) and a **severity × decision** cross-table. The High-severity row is the one to read first: it should show the issues that change headline conclusions (the duplicate invoice IDs, the contradicting payment status, the customer value field, the overstock and the inflow/outflow imbalance). The only High-severity `ACCEPT` should be the foreign-key row (I21), accepted because it found 0 exceptions.

### 6.3 Save the log

In [54]:
Path("docs").mkdir(exist_ok=True)
issue_log.to_csv("docs/dq_issue_log_v2.csv", index=False, encoding="utf-8-sig")
saved = ["docs/dq_issue_log_v2.csv"]
try:
    issue_log.to_excel("docs/dq_issue_log_v2.xlsx", index=False)      # needs openpyxl
    saved.append("docs/dq_issue_log_v2.xlsx")
except ImportError:
    print("openpyxl not installed: CSV saved only (pip install openpyxl for .xlsx)")
print("Saved:", saved)

Saved: ['docs/dq_issue_log_v2.csv', 'docs/dq_issue_log_v2.xlsx']


**Reading the output.** `Saved: [...]` lists the files written to `docs/`. The CSV always works; the Excel copy needs `openpyxl` and is skipped with a message if it is missing. The plan's team format asks for Word documents for sprint records, so paste or import the table into `dq_issue_log.docx` (v2) from these files.

### Task 6: Summary

| Item | Result |
|---|---|
| Issues logged | 28, each with severity, evidence, decision, treatment and owner |
| **FIX** | Capacity, dimensions, postal codes, derived payment status, recomputed customer value |
| **FLAG** | Adjustment direction, `current_stock` differences, reused invoice and payment IDs, overpayments, cancelled-order delivery dates, overstock, inflow/outflow imbalance |
| **ACCEPT** | Date types, cancelled-PO blanks, ledger consistency, finance arithmetic, valid foreign keys, region coverage and master-data notes |
| **EXCLUDE** | Cancelled purchase orders (lead-time measures) and cancelled sales orders (revenue measures), via flag columns |
| Open questions | Adjustment direction, cause of overpayments, genuine `current_stock` differences |

**Status: COMPLETED.** The three open questions need an answer from the programme coordinator or the official Fields Documentation, which was requested in Week 1.

---
## 7. Week 2 wrap-up

### 7.1 Overall summary

| Task | Outcome | Status |
|---|---|---|
| 1. Type casting and standardisation | Capacity, dimensions, volume, postal codes standardised; dates and labels reviewed | Completed |
| 2. Stock reconstruction | Ledger consistent for all 180 pairs; adjustment direction and a few `current_stock` differences flagged | Completed, discrepancies flagged |
| 3. Financial reconciliation | Line, GST and header arithmetic clean; invoice, payment and customer-value exceptions flagged | Completed, exceptions flagged |
| 4. Integrity and business rules | All 17 foreign keys valid; reused invoice and payment IDs flagged | Completed |
| 5. Automation and export | `src/cleaning.py`, pytest suite, 12 Parquet files | Completed once tests pass |
| 6. Issue log | 28 issues, each with a decision | Completed |

### 7.2 What the data is telling us

1. **The structure is sound.** All foreign keys resolve and the order-level money is internally consistent. Later joins will not silently lose rows.
2. **The weak spots are identifiers and summary fields**, not the transactions: re-issued `invoice_id` and `payment_id` values, a stale `payment_status` and a stale `customers.total_purchase_value`.
3. **The ledger is consistent but one-sided.** Stock coming in is about 18 times stock going out, and current stock sits above the maximum level in the product-branch pairs. This is the central business story for Weeks 5 and 10, not a cleaning problem.
4. **Three questions need a business answer:** the direction of adjustments, the cause of overpayments and the genuine `current_stock` differences.

### 7.3 Sprint review (draft for `sprint_review.docx`)

| Catalogue task | What was delivered | Where |
|---|---|---|
| A2 Data Cleaning & Quality Improvement | Standardised tables, reusable pipeline, Parquet layer | Tasks 1 and 5 |
| A5 Data Validation & Consistency Checks | Stock, financial and rule checks with documented results | Tasks 2, 3 and 4 |
| G4 Data Integrity & Error Detection | Key and relationship checks, duplicate-ID investigation, automated tests, issue log with decisions | Tasks 4, 5 and 6 |

### 7.4 Retrospective (draft for `retro.docx`; edit to match the team discussion)

| Keep | Improve | Try |
|---|---|---|
| Order-independent stock test, which removed a false "179 of 180 mismatch" | Do not join on a key before proving it is unique (`invoice_id` double-counted payments) | Ask the coordinator for the Fields Documentation to settle adjustment direction and ID meaning |
| Labelling uncertain invoices *Ambiguous* instead of guessing | Generate the numbers in result notes from live output rather than typing them | Add a sequence column or timestamp to the ledger request for the next data drop |
| Flag columns instead of deleting rows | Run the notebook top to bottom before writing a status line | Pin known issues in tests from the start so growth is caught early |
| Known-issue tests pinned to recorded sizes | | Join invoices on `so_id` and review `payment_id` handling with the data scientist in Week 3 |

### 7.5 Definition of done (Week 2)

| Criterion | Status |
|---|---|
| Cleaning pipeline re-runs end-to-end from raw with one command | Done: `python -m src.cleaning` |
| All tests pass | Confirm in the pytest cell (section 5.7) |
| Every known issue has a decision and rationale | Done: section 6 |
| Cleaned Parquet layer in `data/processed/` | Done: section 5.4 |
| Issue log v2 and validation report as Word documents | To do outside the notebook: CSV and XLSX are in `docs/` |
| `sprint_review.docx`, `retro.docx`, sprint notes, README entry, tag `week-02`, portal submission | To do outside the notebook |

### 7.6 Hand-off to Week 3 (star schema and features)

* Join invoices to orders on **`so_id`**. Carry `invoice_id_duplicated` into the fact table.
* Keep the 393 ambiguous invoices in `fact_invoice_payment` with their ambiguity flag. Do not attribute their payments to one invoice.
* Use `payment_status_derived`, not the stored `payment_status`.
* Use `recalc_purchase_value_with_gst`, not `total_purchase_value`.
* Exclude `is_cancelled` rows from revenue, invoice, lead-time and order-to-delivery measures.
* Use branch-level reorder, safety and max parameters from `inventory_master`.

---
## Appendix A: Run every Task 1–4 check in one go

An optional helper that runs all Task 1–4 checks and returns the results in a single dictionary. It is not needed if you ran the cells above, and it is a convenient way to re-check everything after a data refresh.

In [55]:
def run_week2_checks(con) -> dict:
    """Run every Task 1-4 check and return the results in one dictionary."""
    r = {}
    # Task 1
    r["branches_clean"] = clean_branches(con)
    r["products_clean"] = clean_products(con)
    r["task1_validation"] = validate_task1_outputs(r["branches_clean"], r["products_clean"])
    r["date_check"] = date_column_check(con)
    r["po_status_vs_received"] = po_status_vs_received_date(con)
    r["categorical_check"] = categorical_check(con)
    r["categorical_inconsistencies"] = categorical_inconsistencies(r["categorical_check"])
    # Task 2
    r["stock_movement_summary"] = stock_movement_summary(con)
    r["stock_test"] = stock_adjustment_test(con)
    r["stock_test_summary"] = summarise_stock_test(r["stock_test"])
    # Task 3
    r["sales_line_check"] = line_arithmetic_check(con, "sales")
    r["purchase_line_check"] = line_arithmetic_check(con, "purchase")
    r["sales_header_vs_lines"] = header_vs_lines(con, "sales")
    r["purchase_header_vs_lines"] = header_vs_lines(con, "purchase")
    r["invoice_payment"] = invoice_payment_reconciliation(con)
    r["payment_crosstab"] = payment_status_crosstab(r["invoice_payment"])
    r["customer_recon"] = customer_purchase_reconciliation(con)
    r["customer_recon_summary"] = summarise_customer_reconciliation(r["customer_recon"])
    # Task 4
    r["fk"] = validate_foreign_keys(con)
    r["pk"] = validate_primary_keys(con)
    r["duplicates"] = {}
    for table, key in (("invoices", "invoice_id"), ("payments", "payment_id")):
        det = duplicate_key_details(con, table, key)
        r["duplicates"][table] = {"details": det, **classify_duplicate_keys(det, key)}
    r["po_checks"] = cancelled_po_checks(con)
    r["delivery_checks"] = delivery_date_checks(con)
    r["region_checks"] = customer_region_checks(con)
    r["task4_summary"] = build_task4_summary(
        r["fk"], r["pk"], r["po_checks"], r["delivery_checks"], r["region_checks"])
    return r

**Reading the output.** No output: the function is only defined. Uncomment the two lines below to run it.

In [56]:
results = run_week2_checks(con)
results["task4_summary"]

,Part,Check,Exceptions,Status
0,1a,Foreign keys: orphan or null keys (17 relationships),0,PASSED
1,1b,Primary keys: duplicates in the 10 other tables,0,PASSED
2,1c,Primary key duplicates: invoices.invoice_id,197,FLAGGED
3,1d,Primary key duplicates: payments.payment_id,202,FLAGGED
4,2a,Cancelled POs with a received_date,0,PASSED
5,2b,Active POs missing a received_date,0,PASSED
6,3a,Deliveries before order date,0,PASSED
7,3b,Cancelled orders carrying a delivery date,1967,FLAGGED
8,4,Customers in regions with no local branch,22,REVIEW


**Reading the output.** If run, `results["task4_summary"]` shows the same scorecard as in section 4. The dictionary also holds every other table from Tasks 1–4 under its own name (for example `results["stock_test_summary"]`, `results["invoice_payment"]`).

---

**Prepared by:** Keren Oyem · Data Analyst · CadetX Virtual Work Experience · Heavy Supplier, Inventory & Warehouse Analytics